# Flux API Datatourisme
https://www.datatourisme.fr/

In [1]:
import zipfile
import json
from collections import Counter
from datetime import datetime
import pandas as pd
from typing import Any
import ast
import re
from pathlib import Path
import numpy as np

# 1. CHECK rapides data-safe
À l’issue de ces contrôles (structure, volume, typologie, géolocalisation et fraîcheur), le flux Datatourisme peut être considéré comme sain et prêt à être intégré dans un pipeline de transformation et de valorisation.

### 1.1. Check volume réel de POI

In [2]:
"""
Identification des fichiers POI dans le flux Datatourisme
pour détecter toute anomalie de volume avant de lancer des traitements plus coûteux.
Objectif :
- Lister uniquement les objets métiers (Points of Interest)
- Évaluer rapidement le volume réel de données à traiter
- Disposer d’un indicateur simple pour détecter une anomalie de flux (ex : flux vide ou incomplet)
"""
zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"

# Récupération de la liste complète des fichiers présents dans l’archive
# Les POI sont stockés dans le dossier "objects/" et chaque POI correspond
object_files = list_object_files(zip_path)

# Affichage du nombre total de POI détectés dans le flux
# Ce chiffre sert de contrôle rapide du volume et peut être comparé
# aux runs précédents pour détecter toute variation anormale
print("Nombre total de POI :", len(object_files))

Nombre total de POI : 419714


### 1.2. Check répartition par type (@type)

In [3]:
"""
Objectif : chaque Point of Interest est décrit par un ou plusieurs types. Avant toute transformation ou scoring,
il est crucial de vérifier que la répartition des types est cohérente avec les attentes métier.
- Vérifier que les grands types Datatourisme sont bien présents
- Détecter rapidement une anomalie métier (ex: Restaurants absents)
- Faire un sanity check avant transformation / scoring
Approche :
- Lecture directe depuis le ZIP (sans extraction)
- Échantillonnage (20 000 POI suffisent pour ce check)
- Comptage des types avec Counter
"""
zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"
type_counter = Counter()

with zipfile.ZipFile(zip_path) as z:
    
    # On parcourt un échantillon (rapide et suffisant)
    for name in object_files[:20_000]:
        
        # Lecture du JSON directement depuis l'archive
        data = json.loads(z.read(name))
        
        # Le champ @type peut être une string ou une liste
        types = data.get("@type", [])
        
        if isinstance(types, str):
            types = [types]
            
        # Mise à jour du compteur
        type_counter.update(types)

# Affichage des types les plus fréquents
type_counter.most_common(10)

[('PointOfInterest', 20000),
 ('PlaceOfInterest', 16784),
 ('schema:Accommodation', 4416),
 ('schema:LodgingBusiness', 4416),
 ('Accommodation', 4416),
 ('schema:LocalBusiness', 3822),
 ('schema:FoodEstablishment', 3367),
 ('FoodEstablishment', 3367),
 ('CulturalSite', 3292),
 ('RentalAccommodation', 3220)]

### 1.3. Check blog-ready

In [4]:
"""
Inspection de la structure d’un POI Datatourisme : Cette inspection permet de constater que certains champs,
comme isLocatedAt, peuvent varier en structure (liste ou objet) selon les sources de données amont.
Objectif :
- Explorer le schéma réel d’un objet Datatourisme
- Identifier les champs disponibles au niveau racine
- Comprendre la structure des champs imbriqués (ex: isLocatedAt)
Cette étape est essentielle pour :
- adapter les règles de qualité
- écrire un code robuste face aux variations de schéma
- éviter les hypothèses incorrectes sur la structure des données
"""

zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"

with zipfile.ZipFile(zip_path) as z:

    # Sélection d’un POI représentatif (premier fichier du flux)
    # Un seul objet suffit pour explorer la structure générale
    first = object_files[0]

    # Lecture du fichier JSON directement depuis l’archive
    data = json.loads(z.read(first))

# Affichage des premières clés au niveau racine
# Permet d’identifier rapidement les champs principaux du modèle Datatourisme
print("Keys top-level:", list(data.keys())[:30])

# Affichage du ou des types métier associés au POI
# (@type peut contenir plusieurs valeurs)
print("\n@type:", data.get("@type"))

# Vérification du type Python du champ isLocatedAt
# (dict, list ou None selon les sources amont)
print("\nisLocatedAt type:", type(data.get("isLocatedAt")))

# Aperçu partiel du contenu de isLocatedAt
# Utile pour comprendre l’imbrication réelle sans surcharger l’affichage
print("\nisLocatedAt (preview):", str(data.get("isLocatedAt"))[:800])

Keys top-level: ['@id', 'dc:identifier', '@type', 'rdfs:comment', 'rdfs:label', 'availableLanguage', 'hasArchitecturalStyle', 'hasAudience', 'hasBeenCreatedBy', 'hasBeenPublishedBy', 'hasClientTarget', 'hasContact', 'hasDescription', 'hasMainRepresentation', 'hasRepresentation', 'hasReview', 'hasTheme', 'hasTranslatedProperty', 'isLocatedAt', 'lastUpdate', 'lastUpdateDatatourisme', 'reducedMobilityAccess']

@type: ['ArcheologicalSite', 'CulturalSite', 'PlaceOfInterest', 'PointOfInterest']

isLocatedAt type: <class 'list'>

isLocatedAt (preview): [{'@id': 'https://data.datatourisme.fr/6b3cdaa5-b0fe-3ea4-9a63-c8d68bdb01b1', 'schema:address': [{'@id': 'https://data.datatourisme.fr/3d59a950-9f44-321a-a399-a3339f1955f9', 'schema:addressLocality': 'Sarreinsming', 'schema:postalCode': '57905', '@type': ['schema:PostalAddress', 'PostalAddress'], 'hasAddressCity': {'@id': 'kb:57633', '@type': ['City'], 'rdfs:label': {'de': ['Sarreinsming'], 'pt': ['Sarreinsming'], 'en': ['Sarreinsming'], 'it': 

In [5]:
"""
Exploration détaillée de la structure de localisation d’un POI : permet de gérer les variations de schéma observées
dans le flux Datatourisme (geo au niveau isLocatedAt ou address). Les informations géographiques (schema:geo) peuvent
être présentes ou non dans le champ "isLocatedAt". Cette inspection permet identifier les chemins réellement utilisés
dans le flux et d’écrire un code capable de gérer ces variations de manière robuste.
Objectif :
- Comprendre précisément la structure du champ isLocatedAt
- Identifier où se trouvent les informations d’adresse
- Localiser le champ géographique (schema:geo) dans la hiérarchie
"""

zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"

with zipfile.ZipFile(zip_path) as z:

    # Lecture d’un POI depuis l’archive ZIP
    data = json.loads(z.read(object_files[0]))

# Récupération des informations de localisation
# isLocatedAt est généralement une liste (même lorsqu’un seul élément est présent)
locs = data.get("isLocatedAt", [])

# Parcours des différentes localisations associées au POI
for i, loc in enumerate(locs):

    # Affichage des clés disponibles au niveau isLocatedAt
    print(f"\n--- isLocatedAt[{i}] keys:", loc.keys())

    # Récupération des adresses associées à la localisation
    # schema:address peut également être une liste
    addresses = loc.get("schema:address", [])

    # Parcours des adresses pour inspecter leur contenu
    for j, addr in enumerate(addresses):

        # Affichage des clés de l’objet adresse
        print(f"  schema:address[{j}] keys:", addr.keys())

        # Aperçu du champ schema:geo (peut être absent selon les sources)
        print("  schema:geo preview:", addr.get("schema:geo"))


--- isLocatedAt[0] keys: dict_keys(['@id', 'schema:address', 'schema:geo', '@type', 'petsAllowed'])
  schema:address[0] keys: dict_keys(['@id', 'schema:addressLocality', 'schema:postalCode', '@type', 'hasAddressCity'])
  schema:geo preview: None


In [6]:
"""
Localiser précisément les coordonnées dans un POI afin d'identifier si le champ schema:geo est présent directement
sous isLocatedAt ou imbriqué dans schema:address, et d’en inspecter le contenu réel
Objectif :
- Identifier où se trouvent réellement les coordonnées géographiques dans la structure isLocatedAt
- Gérer les variantes de schéma Datatourisme
Ce script affiche :
- les clés disponibles
- la présence (ou non) de schema:geo
- le contenu exact des coordonnées si elles existent
"""

zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"

with zipfile.ZipFile(zip_path) as z:
    data = json.loads(z.read(object_files[0]))

locs = data.get("isLocatedAt", [])

for i, loc in enumerate(locs):
    print(f"\n=== isLocatedAt[{i}] ===")
    print("Keys :", loc.keys())

    # --- Cas 1 : schema:geo directement sous isLocatedAt
    geo = loc.get("schema:geo")
    if geo is not None:
        print("GEO trouvé directement sous isLocatedAt")
        print("schema:geo :", geo)
        print("latitude  :", geo.get("schema:latitude"))
        print("longitude :", geo.get("schema:longitude"))
    else:
        print("→ Pas de GEO direct sous isLocatedAt")

    # --- Cas 2 : schema:geo sous schema:address
    addresses = loc.get("schema:address", [])
    for j, addr in enumerate(addresses):
        print(f"  --- schema:address[{j}] ---")
        print("  Keys :", addr.keys())

        geo_addr = addr.get("schema:geo")
        if geo_addr is not None:
            print("GEO trouvé sous schema:address")
            print("schema:geo :", geo_addr)
            print("latitude  :", geo_addr.get("schema:latitude"))
            print("longitude :", geo_addr.get("schema:longitude"))
        else:
            print("Pas de GEO sous cette adresse")


=== isLocatedAt[0] ===
Keys : dict_keys(['@id', 'schema:address', 'schema:geo', '@type', 'petsAllowed'])
GEO trouvé directement sous isLocatedAt
schema:geo : {'@id': 'https://data.datatourisme.fr/826f9371-09b1-38b6-aa7e-f82c79e310a5', 'schema:latitude': '49.0981135933993', 'schema:longitude': '7.13630744688794', '@type': ['schema:GeoCoordinates']}
latitude  : 49.0981135933993
longitude : 7.13630744688794
  --- schema:address[0] ---
  Keys : dict_keys(['@id', 'schema:addressLocality', 'schema:postalCode', '@type', 'hasAddressCity'])
Pas de GEO sous cette adresse


### 1.4. Check GEO (lat/long)

In [7]:
"""
CHECK DATA-SAFE — Géolocalisation (latitude/longitude) peuvent être positionnées à différents niveaux de la hiérarchie isLocatedAt.
il est donc nécessaire de détecter et normaliser ces structures avant toute exploitation géographique.
Objectif :
- Vérifier la proportion de POI géocodés (cartographiables)
- Détecter si un flux a un problème de géométrie (beaucoup de POI sans coords)
Approche :
- Lecture directe depuis le ZIP (sans extraction)
- Échantillonnage (10 000 POI suffisent)
- Comptage GEO OK vs GEO manquant
"""
zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"

geo_ok = 0
geo_missing = 0
N = 10_000

with zipfile.ZipFile(zip_path) as z:

    # Parcours d’un échantillon de POI
    for name in object_files[:N]:

        # Lecture du POI directement depuis l’archive ZIP
        data = json.loads(z.read(name))

        geo_found = False

        # Parcours des localisations associées au POI
        for loc in data.get("isLocatedAt", []):

            # Cas 1 — Coordonnées directement sous isLocatedAt
            geo = loc.get("schema:geo")
            if isinstance(geo, dict):
                lat = geo.get("schema:latitude")
                lon = geo.get("schema:longitude")
                if lat is not None and lon is not None:
                    geo_found = True
                    break

            # Cas 2 — Coordonnées imbriquées dans schema:address
            for addr in loc.get("schema:address", []):
                geo = addr.get("schema:geo")
                if isinstance(geo, dict):
                    lat = geo.get("schema:latitude")
                    lon = geo.get("schema:longitude")
                    if lat is not None and lon is not None:
                        geo_found = True
                        break

            if geo_found:
                break

        # Mise à jour des compteurs de qualité
        if geo_found:
            geo_ok += 1
        else:
            geo_missing += 1

# Résumé du check GEO
total = geo_ok + geo_missing
print(f"GEO OK : {geo_ok}/{total} ({geo_ok/total:.1%})")
print(f"GEO manquant : {geo_missing}/{total} ({geo_missing/total:.1%})")

GEO OK : 10000/10000 (100.0%)
GEO manquant : 0/10000 (0.0%)


### 1.5. Check Fraîcheur du flux

In [8]:
"""
CHECK DATA-SAFE — Fraîcheur des données Datatourisme :
La vérification de la fraîcheur des données confirme que le flux Datatourisme consommé correspond bien au dernier run du fournisseur.
La présence d’une date de mise à jour récente (lastUpdateDatatourisme) sur l’ensemble de l’échantillon analysé permet de valider
la fiabilité temporelle du flux avant toute transformation ou exploitation métier.
Objectif :
- Vérifier que le flux est récent
- Confirmer la cohérence avec l'heure de notification
"""
zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"
# Liste des dates de mise à jour collectées
dates = []

# Taille de l’échantillon
# Un sous-ensemble est suffisant pour valider la fraîcheur globale du flux
N = 5_000  # échantillon suffisant

with zipfile.ZipFile(zip_path) as z:

    # Parcours d’un échantillon de POI
    for name in object_files[:N]:

        # Lecture du POI directement depuis l’archive ZIP
        data = json.loads(z.read(name))

        # Extraction de la date de dernière mise à jour Datatourisme
        d = data.get("lastUpdateDatatourisme")
        
        if d:
            # Conversion de la date ISO 8601 en objet datetime Python
            dates.append(datetime.fromisoformat(d.replace("Z", "")))

# Résumé du check de fraîcheur
print("Date min :", min(dates))
print("Date max :", max(dates))
print("Nombre de POI avec date :", len(dates))

Date min : 2025-02-19 05:15:20.143000
Date max : 2026-01-23 07:01:27.832000
Nombre de POI avec date : 5000


# 2. Construire DataFrame avec les colonnes nécessaires depuis ZIP (streaming)

In [9]:
# ------------------------------------------------------------
# 0) paramètres
# ------------------------------------------------------------
zip_path = r"C:\Users\DELL\data\datatourisme\snapshots\2026-01-25_flux250126_complete.zip"

RENAME_MAP = {
    # Identité / typologie
    "@id": "poi_id",
    "@type": "poi_types",
    "rdfs:label.fr": "label_fr",
    "rdfs:label.en": "label_en",

    # Localisation administrative
    "isLocatedAt.schema:address.schema:postalCode": "postal_code",
    "isLocatedAt.schema:address.hasAddressCity.insee": "city_insee",
    "isLocatedAt.schema:address.hasAddressCity.isPartOfDepartment.insee": "dept_insee",
    "isLocatedAt.schema:address.hasAddressCity.isPartOfDepartment.isPartOfRegion.insee": "region_insee",

    # Coordonnées géographiques
    "isLocatedAt.schema:geo.schema:latitude": "latitude",
    "isLocatedAt.schema:geo.schema:longitude": "longitude",

    # Capacité / jauge
    "allowedPersons": "allowed_persons",

    # Avis / rating
    "hasReview.hasReviewValue.schema:ratingValue": "rating_value",

    # Itinéraires / randonnées (si applicable)
    "tourDistance": "tour_distance_m",
    "duration": "duration_min",
    "hasPracticeCondition.duration": "practice_duration_min",
    "durationDays": "duration_days",
    "hasPracticeCondition.durationDays": "practice_duration_days",
    "positiveCumulDifference": "positive_elevation_gain_m",
    "negativeCumulDifference": "negative_elevation_loss_m",

    # Prix
    "offers.schema:priceSpecification.schema:price": "price",
    "offers.schema:priceSpecification.schema:minPrice": "min_price",
    "offers.schema:priceSpecification.schema:maxPrice": "max_price",

    # Style architectural (priorité FR + id/type)
    "hasArchitecturalStyle.rdfs:label.fr": "architectural_style_fr",
    "hasArchitecturalStyle.rdfs:label.en": "architectural_style_en",

    # Description courte (priorité FR + autres langues)
    "hasDescription.shortDescription.fr": "short_desc_fr",
    "hasDescription.shortDescription.en": "short_desc_en",

    # Reviews / notes
    "hasReview.hasReviewValue.rdfs:label.fr": "review_value_label_fr",
    "hasReview.hasReviewValue.rdfs:label.en": "review_value_label_en",
    "hasReview.hasReviewValue.isCompliantWith": "review_compliant_with",

    # Thèmes (catégorisation)
    "hasTheme.rdfs:label.fr": "theme_fr",
    "hasTheme.rdfs:label.en": "theme_en",

    # Localisation — IDs / types
    "isLocatedAt.schema:address.schema:addressLocality": "address_locality",
    "isLocatedAt.schema:address.schema:streetAddress": "street_address",

    # Ville (labels)
    "isLocatedAt.schema:address.hasAddressCity.rdfs:label.fr": "city_label_fr",

    # Département (labels)
    "isLocatedAt.schema:address.hasAddressCity.isPartOfDepartment.rdfs:label.fr": "dept_label_fr",

    # Région (labels)
    "isLocatedAt.schema:address.hasAddressCity.isPartOfDepartment.isPartOfRegion.rdfs:label.fr": "region_label_fr",

    # Pays (labels)
    "isLocatedAt.schema:address.hasAddressCity.isPartOfDepartment.isPartOfRegion.isPartOfCountry.rdfs:label.fr": "country_label_fr",

    # Horaires d'ouverture (attention: listes en réalité)
    "isLocatedAt.schema:openingHoursSpecification.schema:opens": "opens_time",
    "isLocatedAt.schema:openingHoursSpecification.schema:closes": "closes_time",
    "isLocatedAt.schema:openingHoursSpecification.schema:validFrom": "hours_valid_from",
    "isLocatedAt.schema:openingHoursSpecification.schema:validThrough": "hours_valid_through",

    # Updates (dates)
    "lastUpdateDatatourisme": "last_update_datatourisme",

    # Dates (événements / périodes)
    "schema:startDate": "start_date",
    "schema:endDate": "end_date",

    # Practice condition (difficulté, locomotion)
    "hasPracticeCondition.hasDifficultyLevel.rdfs:label.fr": "difficulty_level_fr",
    "hasPracticeCondition.hasLocomotionMode.rdfs:label.fr": "locomotion_mode_fr",

    # Tour type (type de parcours)
    "hasTourType.rdfs:label.fr": "tour_type_fr",

    # url Contact
    "hasContact.foaf:homepage": "contact_homepage",
    "hasRepresentation.ebucore:hasRelatedResource.ebucore:locator": "media_resource_url",
    "hasMainRepresentation.ebucore:hasRelatedResource.ebucore:locator": "main_media_url",}

# ------------------------------------------------------------
# 1) (Re)construire la liste des fichiers POI dans le ZIP
# ------------------------------------------------------------
with zipfile.ZipFile(zip_path) as z:
    object_files = [
        n for n in z.namelist()
        if n.startswith("objects/") and n.endswith(".json")]

print("Nombre de POI dans le zip :", len(object_files))

# ------------------------------------------------------------
# 2) Fonction "data-safe" : extraire une valeur à partir d'un chemin
#    - gère dict / list / None
#    - si list : prend le 1er élément
# ------------------------------------------------------------
def _first_if_list(x: Any) -> Any:
    """Si x est une liste non vide, retourne son premier élément, sinon x."""
    if isinstance(x, list):
        return x[0] if len(x) > 0 else None
    return x

def get_by_path(obj: Any, path: str, sep: str = ".") -> Any:
    """
    Récupère une valeur dans un JSON à partir d'un chemin 'a.b.c'.
    - Si un niveau est une liste, on prend le premier élément.
    - Si une clé est absente, retourne None.
    """
    cur = obj
    for key in path.split(sep):
        cur = _first_if_list(cur)
        if cur is None:
            return None
        if isinstance(cur, dict):
            cur = cur.get(key)
        else:
            # cur n'est ni dict ni None (ex: string/int) => chemin impossible
            return None

    # au cas où la feuille est encore une liste (rare), on prend le 1er élément
    cur = _first_if_list(cur)
    return cur

# ------------------------------------------------------------
# 3) Convertir JSON -> "table" (DataFrame) en ne gardant que les colonnes mappées
# ------------------------------------------------------------
def datatourisme_zip_to_df(
    zip_path: str,
    rename_map: dict,
    object_files: list[str] | None = None,
    limit: int | None = None,
    column_order: list[str] | None = None,
) -> pd.DataFrame:
    """
    Lit le ZIP Datatourisme et renvoie un DataFrame :
    - 1 ligne = 1 POI
    - colonnes = celles de rename_map (renommées)
    - extraction robuste (dict/list) via get_by_path
    - conversions de types "safe"
    - réorganisation des colonnes (si column_order fourni)
    """
    rows = []
    wanted_paths = list(rename_map.keys())

    with zipfile.ZipFile(zip_path) as z:
        files = object_files
        if files is None:
            files = [n for n in z.namelist() if n.startswith("objects/") and n.endswith(".json")]
        if limit is not None:
            files = files[:limit]

        for name in files:
            data = json.loads(z.read(name))
            # Construire une ligne en extrayant UNIQUEMENT les chemins utiles
            row = {rename_map[p]: get_by_path(data, p) for p in wanted_paths}
            rows.append(row)

    df = pd.DataFrame(rows)

    # ----------------------------
    # Conversions de types "safe"
    # ----------------------------
    int_cols = ["postal_code", "city_insee", "dept_insee", "region_insee"]
    for c in int_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    float_cols = [
        "latitude", "longitude",
        "allowed_persons", "rating_value",
        "tour_distance_m", "duration_min", "practice_duration_min",
        "duration_days", "practice_duration_days",
        "positive_elevation_gain_m", "negative_elevation_loss_m",
        "price", "min_price", "max_price",]
    
    for c in float_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # ----------------------------
    # Réorganisation des colonnes
    # ----------------------------
    if column_order is not None:
        existing = [c for c in column_order if c in df.columns]
        rest = [c for c in df.columns if c not in existing]
        df = df[existing + rest]

    return df


# ------------------------------------------------------------
# Ordre recommandé des colonnes (51 colonnes)
# ------------------------------------------------------------
ORDERED_COLS = [
    # A. Identité / clés
    "poi_id", "poi_types",

    # B. Contenu FR
    "label_fr", "label_en", "short_desc_fr", "short_desc_en",

    # C. Coordonnées
    "latitude", "longitude",

    # D. Admin
    "country_label_fr",
    "region_insee", "region_label_fr",
    "dept_insee", "dept_label_fr",
    "city_insee", "city_label_fr",
    "postal_code",

    # E. Adresse
    "address_locality", "street_address",

    # F. Tags
    "theme_fr", "theme_en", "architectural_style_fr", "architectural_style_en",

    # G. Médias
    "main_media_url", "media_resource_url", "contact_homepage",

    # I. Reviews / labels
    "rating_value", "review_value_label_fr", "review_value_label_en", "review_compliant_with",

    # J. Horaires
    "opens_time", "closes_time",
    "hours_valid_from", "hours_valid_through",

    # L. Itinéraires / pratique
    "allowed_persons",
    "difficulty_level_fr", "locomotion_mode_fr", "tour_type_fr",
    "tour_distance_m",
    "duration_min", "practice_duration_min",
    "duration_days", "practice_duration_days",
    "positive_elevation_gain_m", "negative_elevation_loss_m",

    # M. Dates & fraîcheur
    "start_date", "end_date", "last_update_datatourisme",]

# ------------------------------------------------------------
# Exécution
# ------------------------------------------------------------
df_dt = datatourisme_zip_to_df(
    zip_path=zip_path,
    rename_map=RENAME_MAP,
    object_files=object_files,
    limit=None,
    column_order=ORDERED_COLS,)

pd.set_option("display.max_columns", None)
#pd.set_option("display.max_colwidth", None)
print(df_dt.shape)
display(df_dt.head(3))

Nombre de POI dans le zip : 419714
(419714, 50)


,poi_id,poi_types,label_fr,label_en,short_desc_fr,short_desc_en,latitude,longitude,country_label_fr,region_insee,region_label_fr,dept_insee,dept_label_fr,city_insee,city_label_fr,postal_code,address_locality,street_address,theme_fr,theme_en,architectural_style_fr,architectural_style_en,main_media_url,media_resource_url,contact_homepage,rating_value,review_value_label_fr,review_value_label_en,review_compliant_with,opens_time,closes_time,hours_valid_from,hours_valid_through,allowed_persons,difficulty_level_fr,locomotion_mode_fr,tour_type_fr,tour_distance_m,duration_min,practice_duration_min,duration_days,practice_duration_days,positive_elevation_gain_m,negative_elevation_loss_m,start_date,end_date,last_update_datatourisme,price,min_price,max_price
0,https://data.datatourisme.fr/10/000283ba-f94b-...,ArcheologicalSite,Villa gallo-romaine du grosswald,None,Les ruines de la Villa du Grosswald sont en bo...,The ruins of the Villa du Grosswald lie on the...,49.098114,7.136307,France,44,Grand Est,57,Moselle,57633,Sarreinsming,57905,Sarreinsming,None,Antique,Antique,Antique,Antique,https://opendata.sitlor.fr/photos/947/94700124...,https://opendata.sitlor.fr/photos/947/94700124...,None,NaN,Monument Historique,Monument Historique,CulturalSite,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,2025-08-21T04:08:28.719Z,NaN,NaN,NaN
1,https://data.datatourisme.fr/10/0016c366-d21e-...,schema:Accommodation,Meublé - résidence les Vosges - studio n°95,None,Résidence située en centre-ville à proximité d...,Residence located in the town center near the ...,48.002610,6.263993,France,44,Grand Est,88,Vosges,88029,La Vôge-les-Bains,88240,La Vôge-les-Bains,11/13 rue du Général Leclerc,None,None,None,None,None,None,None,3.0,3 étoiles,3 Stars,CampingAndCaravanning,08:00:00,21:00:00,2026-03-15T00:00:00,2026-11-07T23:59:59,2.0,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,2025-12-21T06:05:13.805Z,NaN,NaN,NaN
2,https://data.datatourisme.fr/10/0016e4ef-68b7-...,CulturalSite,Église paroissiale de la nativité,None,Destinée à recevoir les Chevaliers de l'Ordre ...,Intended to receive the Knights of the Teutoni...,49.441251,6.356225,France,44,Grand Est,57,Moselle,57650,Sierck-les-Bains,57480,Sierck-les-Bains,rue de la Tour de l'Horloge,None,None,None,None,None,None,http://www.siercklesbains.fr/,NaN,None,None,None,None,None,None,None,NaN,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,2026-01-02T05:18:38.54Z,NaN,NaN,NaN


# 3. Résumer les valeurs uniques par colonnes

In [10]:
"""
Fonction générique : résumé des valeurs uniques par colonne. Pour chaque colonne :
- type (dtype)
- % de valeurs manquantes
- nombre de valeurs uniques
- aperçu des N premières valeurs unique
"""

def describe_unique_values(
    df: pd.DataFrame,
    max_unique_preview: int = 10,
    max_columns: int | None = None,
    include_object: bool = True,
    include_numeric: bool = True):
    """
    Affiche un résumé des valeurs uniques par colonne :
    - type de la colonne
    - pourcentage de valeurs manquantes
    - nombre de valeurs uniques
    - aperçu des premières valeurs uniques

    Paramètres
    ----------
    df : pd.DataFrame
        DataFrame à analyser
    max_unique_preview : int
        Nombre de valeurs uniques à afficher en aperçu
    max_columns : int | None
        Limiter le nombre de colonnes analysées (None = toutes)
    include_object : bool
        Inclure les colonnes de type object
    include_numeric : bool
        Inclure les colonnes numériques
    """

    cols = df.columns
    if max_columns:
        cols = cols[:max_columns]

    for col in cols:
        dtype = df[col].dtype

        if dtype == "object" and not include_object:
            continue
        if pd.api.types.is_numeric_dtype(dtype) and not include_numeric:
            continue

        total = len(df)
        missing = df[col].isna().sum()
        missing_pct = (missing / total) * 100
        unique_vals = df[col].dropna().unique()

        print("\n-------------------------------------")
        print(f"Colonne : {col}")
        print(f"Type : {dtype}")
        print(f"Valeurs manquantes : {missing_pct:.1f}%")
        print(f"Nombre de valeurs uniques : {len(unique_vals)}")
        print("-------------------------------------")

        display(unique_vals[:max_unique_preview])

#les colonnes object
describe_unique_values(df_dt)


-------------------------------------
Colonne : poi_id
Type : object
Valeurs manquantes : 0.0%
Nombre de valeurs uniques : 419714
-------------------------------------


array(['https://data.datatourisme.fr/10/000283ba-f94b-3bce-8f57-e5c10bec6cd4',
       'https://data.datatourisme.fr/10/0016c366-d21e-3f85-8c8a-a0d6c6d71a67',
       'https://data.datatourisme.fr/10/0016e4ef-68b7-3e79-87c4-5a277e1a43ac',
       'https://data.datatourisme.fr/10/00206603-d2ce-3c61-b93a-f748c9f710a1',
       'https://data.datatourisme.fr/10/002adbf5-395a-39a8-80d4-bddf2beebc2f',
       'https://data.datatourisme.fr/10/002c1ef6-286d-34f8-94ac-86e19982ac36',
       'https://data.datatourisme.fr/10/002d7a0f-de91-3c8c-b635-2019e988bda0',
       'https://data.datatourisme.fr/10/002e824b-a1f7-3100-94a7-352b37fee916',
       'https://data.datatourisme.fr/10/002f97fd-5efb-3da6-b8a3-db09853e2293',
       'https://data.datatourisme.fr/10/0030b2ab-98b9-3904-b6f1-4d681014641b'],
      dtype=object)


-------------------------------------
Colonne : poi_types
Type : object
Valeurs manquantes : 0.0%
Nombre de valeurs uniques : 77
-------------------------------------


array(['ArcheologicalSite', 'schema:Accommodation', 'CulturalSite',
       'schema:FastFoodRestaurant', 'schema:Event', 'schema:Library',
       'olo:OrderedList', 'schema:Park', 'schema:LocalBusiness',
       'schema:FoodEstablishment'], dtype=object)


-------------------------------------
Colonne : label_fr
Type : object
Valeurs manquantes : 0.0%
Nombre de valeurs uniques : 371130
-------------------------------------


...

array([ 0. ,  5. , 14. ,  7.8,  2. , 32. , 10. , 18. , 15. , 40. ])


-------------------------------------
Colonne : min_price
Type : float64
Valeurs manquantes : 80.0%
Nombre de valeurs uniques : 2778
-------------------------------------


array([ 12.9 ,  19.9 ,   8.5 ,  19.  ,  18.  ,   4.  ,  70.  ,   9.95,
        10.  , 140.  ])


-------------------------------------
Colonne : max_price
Type : float64
Valeurs manquantes : 84.5%
Nombre de valeurs uniques : 3000
-------------------------------------


array([  79. ,    7.2,   30. ,   95. ,  100. ,   54. ,  380. ,   25. ,
       2457. ,   78. ])

# 4. Nettoyer + compléter + transformer les valeurs

### 4.1. Détecter et supprimer les doublons

In [11]:
#-------------------------------------------------------
# Bornes géographiques – France métropolitaine
# Utilisées pour filtrer les coordonnées aberrantes
#-------------------------------------------------------
FR_METRO_BOUNDS = {
    "lat_min": 41.0,
    "lat_max": 51.6,
    "lon_min": -5.5,
    "lon_max": 10.0,}

#-------------------------------------------------------
# Nettoyage géographique (lat / lon)
#-------------------------------------------------------
def clean_geo_fr_metro(
    df: pd.DataFrame,
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    bounds: dict = FR_METRO_BOUNDS,
    drop_zero_zero: bool = True,
) -> pd.DataFrame:
    """
    Nettoie la géolocalisation des POI pour la France métropolitaine.
    Étapes :
    1) Convertit latitude / longitude en numérique
    2) Supprime les lignes sans coordonnées (NaN)
    3) Supprime les coordonnées (0,0) si drop_zero_zero=True
       souvent utilisé comme "valeur par défaut" invalide
    4) Conserve uniquement les points situés dans les bornes FR métropole
    Retour :
    - DataFrame nettoyé, prêt pour déduplication
    """
    out = df.copy()

    # Sécurise les types (au cas où ce sont des strings)
    out[lat_col] = pd.to_numeric(out[lat_col], errors="coerce")
    out[lon_col] = pd.to_numeric(out[lon_col], errors="coerce")

    # 1) Supprimer les POI sans coordonnées
    out = out.dropna(subset=[lat_col, lon_col])

    # 2) Supprimer les coordonnées invalides (0,0)
    if drop_zero_zero:
        out = out.loc[~((out[lat_col] == 0) & (out[lon_col] == 0))].copy()

    # 3) Filtrer sur les bornes France métropolitaine
    out = out.loc[
        out[lat_col].between(bounds["lat_min"], bounds["lat_max"], inclusive="both") &
        out[lon_col].between(bounds["lon_min"], bounds["lon_max"], inclusive="both")].copy()

    return out


#-------------------------------------------------------
# Déduplication générique – garder la “meilleure” ligne
#-------------------------------------------------------
def deduplicate_keep_best(
    df: pd.DataFrame,
    key_cols: list[str],
    last_update_col: str = "last_update_datatourisme",
    quality_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Déduplique un DataFrame en conservant la meilleure version par groupe.
    Règles :
    - Les lignes sont regroupées selon key_cols
    - On conserve en priorité :
        1) la plus récente (last_update_datatourisme)
        2) la plus "riche" (le plus de champs non nuls)
    Paramètres :
    - key_cols : colonnes définissant un doublon logique
    - last_update_col : colonne de date de mise à jour
    - quality_cols : colonnes utilisées pour mesurer la richesse
    Retour :
    - DataFrame dédupliqué
    """
    out = df.copy()

    # Convertir la date de mise à jour en datetime (UTC)
    out["_last_update_dt"] = pd.to_datetime(out[last_update_col], errors="coerce", utc=True)

    # Calcul du score de "richesse"
    if quality_cols is None:
        # fallback : toutes colonnes sauf celles de clé + last_update
        excluded = set(key_cols + [last_update_col])
        quality_cols = [c for c in out.columns if c not in excluded]

    out["_quality"] = out[quality_cols].notna().sum(axis=1)

    # Tri : plus récent puis plus riche
    out = out.sort_values(["_last_update_dt", "_quality"], ascending=[False, False])

    # Supprimer les doublons (on garde le premier après tri)
    out = out.drop_duplicates(subset=key_cols, keep="first")

    # Nettoyage des colonnes temporaires
    out = out.drop(columns=["_last_update_dt", "_quality"], errors="ignore")

    return out


#-------------------------------------------------------
# Pipeline complet Datatourisme – France métropolitaine
#-------------------------------------------------------
def dedup_datatourisme_fr_metro(
    df: pd.DataFrame,
    strict: bool = True,
    coord_round: int = 6,
    quality_cols: list[str] | None = None,
) -> tuple[pd.DataFrame, dict]:
    """
    Pipeline complet de nettoyage + déduplication pour Datatourisme (France métropolitaine).
    Étapes :
    1) Nettoyage géographique (NaN, 0,0, hors bornes FR métro)
    2) Déduplication "safe" sur poi_id
    3) (optionnel) Déduplication stricte sur :
       label_fr + latitude + longitude + poi_types
    Paramètres :
    - strict : active la déduplication stricte
    - coord_round : précision géographique (6 ≈ 0,1 m)
    - quality_cols : colonnes utilisées pour scorer la richesse
    Retour :
    - df_clean : DataFrame final dédupliqué
    - report : statistiques avant / après
    """
    report = {}

    def _report(tag: str, d: pd.DataFrame):
        report[tag] = {
            "rows": len(d),
            "dup_poi_id": int(d.duplicated(["poi_id"]).sum()) if "poi_id" in d.columns else None,
            "zero_or_nan_coords": None,}  # déjà filtré ensuite

    # 1) Nettoyage géographique
    d1 = clean_geo_fr_metro(df)
    _report("after_geo_clean", d1)

    # Colonnes utilisées pour évaluer la "richesse" d’un POI
    if quality_cols is None:
        quality_cols = [
            "label_fr", "short_desc_fr", "comment_fr",
            "main_media_url", "media_resource_url",
            "theme_fr", "feature_fr",
            "street_address", "address_locality",
            "contact_homepage",]
        # garde seulement celles qui existent
        quality_cols = [c for c in quality_cols if c in d1.columns]

    # 2) Déduplication SAFE : poi_id
    if "poi_id" in d1.columns:
        d2 = deduplicate_keep_best(
            d1,
            key_cols=["poi_id"],
            last_update_col="last_update_datatourisme",
            quality_cols=quality_cols)
    else:
        d2 = d1.copy()
    _report("after_dedup_poi_id", d2)

    # 3) Déduplication STRICTE (optionnelle)
    if strict:
        d2 = d2.copy()
        d2["_lat_r"] = d2["latitude"].round(coord_round)
        d2["_lon_r"] = d2["longitude"].round(coord_round)

        key_cols = ["label_fr", "_lat_r", "_lon_r"]
        if "poi_types" in d2.columns:
            key_cols.append("poi_types")

        d3 = deduplicate_keep_best(
            d2,
            key_cols=key_cols,
            last_update_col="last_update_datatourisme",
            quality_cols=quality_cols
        ).drop(columns=["_lat_r", "_lon_r"], errors="ignore")
        _report("after_dedup_strict", d3)
        return d3, report

    return d2, report

df_clean, rep = dedup_datatourisme_fr_metro(df_dt, strict=False)
rep

df_clean, rep = dedup_datatourisme_fr_metro(df_dt, strict=True, coord_round=6)
rep

{'after_geo_clean': {'rows': 419673,
  'dup_poi_id': 0,
  'zero_or_nan_coords': None},
 'after_dedup_poi_id': {'rows': 419673,
  'dup_poi_id': 0,
  'zero_or_nan_coords': None},
 'after_dedup_strict': {'rows': 411634,
  'dup_poi_id': 0,
  'zero_or_nan_coords': None}}

In [12]:
# Contrôle de qualité
# s’assurer qu’il ne reste plus de doublons “visuels”
display(df_clean.duplicated(subset=["label_fr", "latitude", "longitude", "poi_types"]).sum())

#POIs supprimés par la dédup stricte
removed = (df_dt.merge(df_clean[["poi_id"]], on="poi_id", how="left", indicator=True).query('_merge == "left_only"'))
removed[["poi_id","label_fr", "poi_types", "latitude","longitude","last_update_datatourisme"]].head(10)

np.int64(0)

,poi_id,label_fr,poi_types,latitude,longitude,last_update_datatourisme
971,https://data.datatourisme.fr/10/0f9b0ca7-9b13-...,Appartement 6 personnes,schema:Accommodation,48.059523,6.886692,2025-12-10T06:02:50.21Z
2376,https://data.datatourisme.fr/10/26dd992c-4a76-...,Appartement 4 personnes,schema:Accommodation,48.006389,6.884722,2025-12-13T06:01:11.91Z
2911,https://data.datatourisme.fr/10/302abf99-70c3-...,Boulangerie-pâtisserie Dudot Metz,schema:LocalBusiness,49.118103,6.176679,2026-01-02T06:01:57.279Z
4624,https://data.datatourisme.fr/10/4cf3602c-1800-...,Appartement 4 personnes,schema:Accommodation,48.071549,6.870521,2025-12-10T06:02:50.21Z
5297,https://data.datatourisme.fr/10/58246584-895c-...,Site verrier de Meisenthal,schema:LocalBusiness,48.965177,7.353087,2026-01-16T05:19:09.637Z
6633,https://data.datatourisme.fr/10/6da681e5-10b8-...,Appartement 4 personnes,schema:Accommodation,48.070892,6.868639,2025-12-10T06:02:50.209Z
6816,https://data.datatourisme.fr/10/70fbb1ee-ce7a-...,Studio 3 personnes,schema:Accommodation,48.070140,6.878937,2025-12-10T06:02:50.21Z
6973,https://data.datatourisme.fr/10/738f483e-abc2-...,Appartement 4 personnes,schema:Accommodation,48.057325,6.851911,2025-12-10T06:02:50.21Z
7139,https://data.datatourisme.fr/10/76ac9c75-ab9a-...,Appartement 4 personnes,schema:Accommodation,48.087641,6.880861,2025-08-06T05:12:56.029Z
7207,https://data.datatourisme.fr/10/77f12d02-29ef-...,Appartement 4 personnes,schema:Accommodation,48.068813,6.833700,2025-12-10T06:02:50.21Z


### 4.2. Créer colonne is_resto

In [13]:
def clean_poi_types(x):
    """
    Nettoie une cellule poi_types :
    - accepte list / string / NaN
    - supprime schema:, olo:, autres namespaces
    - met en minuscule
    - enlève doublons (ordre conservé)
    - retourne toujours une list[str]
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    # si c'est une string type "['schema:Park', 'schema:Event']"
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except Exception:
            x = [x]

    if not isinstance(x, (list, tuple, set)):
        return []

    cleaned = []
    seen = set()

    for t in x:
        if not isinstance(t, str):
            continue

        t = t.strip().lower()

        # enlever namespace (schema:, olo:, etc.)
        if ":" in t:
            t = t.split(":", 1)[1]

        if t and t not in seen:
            seen.add(t)
            cleaned.append(t)

    return cleaned

#Application sur le DataFrame
df_clean["poi_types_clean"] = df_clean["poi_types"].apply(clean_poi_types)

#Vérifications rapides (toujours utiles)
df_clean[["poi_types", "poi_types_clean"]].head()

#value_counts poi_types
type_counts = (df_clean["poi_types_clean"].explode().value_counts().sort_values(ascending=False))
pd.set_option("display.max_rows", None)
type_counts.head(100)

poi_types_clean
accommodation                      117399
localbusiness                       75741
event                               47280
foodestablishment                   45992
orderedlist                         21983
culturalsite                        21559
placeofinterest                     18309
product                             14320
landform                             7327
church                               4866
fastfoodrestaurant                   4557
cityheritage                         3736
library                              3390
park                                 3360
civicstructure                       2251
convenientservice                    1918
cafeorcoffeeshop                     1779
castle                               1611
bakery                               1413
businessevent                        1318
movietheater                         1223
chapel                               1196
amusementpark                        1056
archeologicalsite 

In [14]:
#-----------------------------------
# Créer colonne is_resto : Normalise un type Datatourisme
#-----------------------------------
def _norm_type(t: str) -> str:
    """
    Normalise un type Datatourisme / Schema.org.
    Objectif :
    - rendre les comparaisons robustes
    - ignorer les préfixes RDF (ex: 'schema:')
    - éviter les problèmes de casse ou d'espaces

    Exemples :
    - 'schema:Restaurant'  -> 'restaurant'
    - 'FastFoodRestaurant'-> 'fastfoodrestaurant'
    - 'schema:Restaurant '-> 'restaurant'
    """
    return str(t).lower().replace("schema:", "").strip()


# Ensemble des types considérés comme des restaurants.
# Ces POIs seront exclus du mode Prime (restaurants injectés via TripAdvisor).
RESTAURANT_TYPES = {"restaurant", "fastfoodrestaurant", "foodestablishment"}


#-----------------------------------
# Normalise un label pour la détection de mots-clés
#-----------------------------------
def _norm_label(s: str) -> str:
    """
    Normalise un label pour la détection de mots-clés :
    - lowercase
    - strip
    - espaces multiples -> 1 espace
    """
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


# Préfix "forts" : très fiables (si le label commence comme ça, on considère resto)
RESTO_LABEL_PREFIXES = ("restaurant", "resto", "pizzeria", "crêperie", "creperie",
                        "bar-restaurant", "bar restaurant", "café-restaurant", "tacos",
                        "cafe-restaurant", "cafe restaurant", "snack", "kebab", "grill")

# Mots-clés : un peu moins stricts (utile quand le label contient "restaurant / brasserie" au milieu)
RESTO_LABEL_KEYWORDS = ("restaurant", "pizzeria", "bistrot", "snack",
                        "kebab", "tacos", "grill", "burger", "sushi")


#-----------------------------------
# Détecte un restaurant via la colonne poi_types (string)
#-----------------------------------
def compute_is_resto_from_str(poi_type: str) -> bool:
    """
    Détecte un restaurant via la colonne poi_types (string).
    Règle :
    - si poi_types normalisé ∈ RESTAURANT_TYPES -> True
    - sinon False
    """
    if pd.isna(poi_type):
        return False
    t = _norm_type(poi_type)
    return t in RESTAURANT_TYPES

    
#-----------------------------------
# Détecte un restaurant via le label_fr
#-----------------------------------
def compute_is_resto_from_label(label_fr: str) -> bool:
    """
    Détecte un restaurant via le label_fr (heuristique).
    Cas d'usage : Datatourisme typé café / salon de thé mais le label dit clairement "Restaurant ...".
    Règle :
    - True si label commence par un préfix fort (Restaurant..., Brasserie..., etc.)
    - ou si le label contient un mot-clé restaurant (mot entier)
    """
    s = _norm_label(label_fr)
    if not s:
        return False

    # 1) Préfix forts
    if s.startswith(RESTO_LABEL_PREFIXES):
        return True

    # 2) Mots-clés (match mot entier pour limiter les faux positifs)
    for kw in RESTO_LABEL_KEYWORDS:
        if re.search(rf"\b{re.escape(kw)}\b", s):
            return True

    return False


# ----------------------------
# Création des colonnes resto
# ----------------------------
df_clean["is_resto_type"] = df_clean["poi_types"].apply(compute_is_resto_from_str)
df_clean["is_resto_label"] = df_clean["label_fr"].apply(compute_is_resto_from_label)
df_clean["is_resto"] = df_clean["is_resto_type"] | df_clean["is_resto_label"]

# ----------------------------
# CONTRÔLE QUALITÉ (sans mask undefined)
# ----------------------------
print("Distribution is_resto :")
display(df_clean["is_resto"].value_counts(dropna=False))

# 1) Exemples détectés via TYPE (très fiable)
print("\nExemples is_resto_type == True :")
display(df_clean.loc[df_clean["is_resto_type"], ["label_fr", "poi_types", "is_resto_type", "contact_homepage", "is_resto"]].head(10))

# 2) Exemples détectés via LABEL seulement (à surveiller)
print("\nExemples is_resto_label == True & is_resto_type == False :")
mask_label_only = df_clean["is_resto_label"] & ~df_clean["is_resto_type"]
display(df_clean.loc[mask_label_only, ["label_fr", "poi_types", "is_resto_label", "contact_homepage", "is_resto"]].head(15))

Distribution is_resto :


is_resto
False    358725
True      52909
Name: count, dtype: int64


Exemples is_resto_type == True :


,label_fr,poi_types,is_resto_type,contact_homepage,is_resto
311729,"Pizzéria ""Annabella""",schema:FoodEstablishment,True,None,True
312172,"Restaurant ""Le Diapason""",schema:FoodEstablishment,True,None,True
312327,"Brasserie ""Seven Café""",schema:FoodEstablishment,True,None,True
312557,"Restaurant ""Chez Toshi""",schema:FoodEstablishment,True,None,True
312910,"Brasserie ""Le Cardinal""",schema:FoodEstablishment,True,None,True
313411,L'Odyssée,schema:FoodEstablishment,True,None,True
313425,"Restaurant ""Au Dernier Sou""",schema:FoodEstablishment,True,None,True
313506,"Pizzeria ""Le Rossini""",schema:FoodEstablishment,True,None,True
311764,"Restaurant ""Basilic & Co""",schema:FastFoodRestaurant,True,https://commande.basilic-and-co.com/restaurant...,True
311946,"Restaurant Pizzéria ""Les Colonnes""",schema:FoodEstablishment,True,None,True



Exemples is_resto_label == True & is_resto_type == False :


,label_fr,poi_types,is_resto_label,contact_homepage,is_resto
311939,"Restaurant ""La Principauté""",schema:Accommodation,True,None,True
314091,"Restaurant ""Le Dalang""",schema:Accommodation,True,None,True
312529,"Restaurant ""Inn Design""",schema:Accommodation,True,http://hotel-inn-sedan.fr/,True
311707,Restaurant Presqu'île de Chooz,schema:Accommodation,True,http://www.ile-chooz.fr/,True
312838,"Restaurant ""Le Saint Michel""",schema:Accommodation,True,http://www.le-saint-michel.fr/,True
314145,"Restaurant ""Kyriad"" Sedan",schema:Accommodation,True,http://sedan.kyriad.com/fr-fr,True
312214,"Restaurant ""Couleurs Sud""",schema:Accommodation,True,http://www.hotel-charleville-mezieres.com/,True
313334,"Restaurant ""Le Campanile""",schema:Accommodation,True,None,True
362648,Crêperie Pause K'fée,schema:LocalBusiness,True,None,True
363776,Boucherie Restaurant Kolifrath Yann,schema:LocalBusiness,True,http://www.boucheriekolifrath.fr/,True


### 4.3. Créer colonnes label :
"is_label_incontournable", "is_label_famille", "is_label_handicap", "is_label_gastronomie", "is_label_hebergement", "is_label_green", "is_label_artisanat", "is_label_randonnee", "is_label_green"

In [15]:
df_clean["review_value_label_fr"].value_counts()

review_value_label_fr
3 étoiles                                                                  19938
2 étoiles                                                                  13399
3 épis / Confort                                                            6992
Accueil Vélo                                                                5802
4 étoiles                                                                   5562
Gîtes de France                                                             3629
1 étoile                                                                    3553
2 épis                                                                      3395
Vignobles & Découvertes                                                     3077
Qualité Tourisme                                                            2701
Monument Historique                                                         2495
Bienvenue à la ferme                                                        2142
Agricu

In [16]:
def add_label_flags(
    df: pd.DataFrame,
    label_col: str = "review_value_label_fr"
) -> pd.DataFrame:
    """
    Crée des colonnes booléennes de flags de labels à partir de `review_value_label_fr`.
    Chaque colonne correspond à une famille de labels (incontournable, famille,
    handicap, hébergement, gastronomie, artisanat, randonnée, green).
    - True : le label appartient à la famille
    - False : sinon ou si NaN
    Aucun impact sur le scoring : ces colonnes sont des signaux explicatifs / UI.
    """
    # --- Définition des familles de labels ---
    LABEL_GROUPS = {
        "is_label_incontournable": {"Patrimoine Mondial UNESCO", "Monument Historique",
                                    "Grand Site de France", "Musée de France",
                                    "Architecture contemporaine remarquable",
                                    "Entreprise du Patrimoine Vivant, EPV",
                                    "Maison des Illustres", "Jardin remarquable",
                                    "Jardin protégé Monument Historique",
                                    "Plus Beaux Villages de France",
                                    "Petites cités de caractère",
                                    "Ville ou Pays d'art et d'histoire",
                                    "Site protégé","Parc Naturel National",
                                    "Réserve naturelle nationale",},

        "is_label_famille": {"Famille plus",},

        "is_label_handicap": {"Tourisme & Handicap auditif", "Tourisme & Handicap mental",
                              "Tourisme & Handicap visuel", "Tourisme & Handicap moteur",},

        "is_label_hebergement": {"5 étoiles", "4 étoiles", "3 étoiles", "3 épis / Confort",
                                 "4 épis / Premium", "5 épis / Luxury ", "3 Clés", "4 Clés", "5 Clés",
                                 "Gîtes de France", "Chambre d'hôtes référence", "Hébergement Pêche",
                                 "Hôtel Elégance", "Hôtel Essentiel", "Auberge de Village", "Hôtel Cosy",
                                 "Camping Qualité",},

        "is_label_gastronomie": {"Maître Restaurateur", "Gault&Millau", "Sélection Michelin",
                                 "Restaurant Gourmand", "Restaurateur de Qualité", "Restaurant de Terroir",
                                 "Bottin Gourmand", "Restaurant Savoureux", "Food Index for Good",
                                 "Le Fooding", "Maître Cuisinier de France", "Table gastronomique",
                                 "Membre du Conservatoire Grand Sud des Cuisines de Terroir",
                                 "Table de terroir", "Tables et auberges de France", "Ecotable",
                                 "Sites Remarquables du Goût", "Bistrot de Pays", "Membre Gourméditerrannée",
                                 "Vélo & Fromage", "3 étoiles Michelin"},

        "is_label_artisanat": {"Vignobles & Découvertes", "Agriculture Biologique", "Bienvenue à la ferme",
                               "Vignerons Indépendants de France", "Accueil Paysan", "Agriculture raisonnée",
                               "Artisans Militants de la Qualité", "Domaine du conservatoire de l'espace littoral",
                               "Teritoria",},

        "is_label_randonnee": {"Plan Départemental des Itinéraires de Promenade et de Randonnée, PDIPR",
                               "Itinéraire de promenade et de randonnée, PR", "Fédération Française de Randonnée",
                               "Itinéraire de sentier de grande randonnée, GR",
                               "Itinéraire de sentier de grande randonnée de pays GRP",},

        "is_label_green": {"4 fleurs", "3 fleurs", "5 fleurs", "Parc Naturel Régional", "Qualité Tourisme",
                           "Pavillon bleu", "Espace Naturel Sensible", "Ecolabel européen", "Valeurs Parc Naturel Régional",
                           "Valeur", "Démarche Tourisme Responsable", "Station Verte", "Esprit parc national",
                           "Balade à roulette, BR", "Clé Vacances", "Véloroute", "Destination d'excellence",
                           "Zone naturelle d'intérêt écologique, faunistique et floristique, ZNIEFF",
                           "Réserve naturelle régionale", "Plus Beaux Détours de France", "Site VTT-FFC",
                           "Natura 2000", "City Break Confort", "Aire Naturelle", "Relais & châteaux",
                           "Fleurs de Soleil", "Eco Jardin", "ISO 20121", "City Break Premium",
                           "Accueil chemins de Compostelle en France", "Grande Traversée VTT-FFC",},}

    # Application des flags
    out = df.copy()

    for col_name, labels in LABEL_GROUPS.items():
        out[col_name] = out[label_col].isin(labels)

    return out

# contrôle qualité
df_clean = add_label_flags(df_clean)
df_clean[["label_fr", "review_value_label_fr",
        "is_label_incontournable", "is_label_famille", "is_label_handicap",
        "is_label_gastronomie", "is_label_hebergement",
        "is_label_green", "is_label_artisanat",
        "is_label_randonnee", "is_label_green"]].head(10)

,label_fr,review_value_label_fr,is_label_incontournable,is_label_famille,is_label_handicap,is_label_gastronomie,is_label_hebergement,is_label_green,is_label_artisanat,is_label_randonnee,is_label_green
312421,Le Monument aux Morts du Maquis des Manises et...,None,False,False,False,False,False,False,False,False,False
312749,Sentier découverte du Vieil-Étang de Bairon,None,False,False,False,False,False,False,False,False,False
311729,"Pizzéria ""Annabella""",None,False,False,False,False,False,False,False,False,False
311896,Le Château de Charbogne,None,False,False,False,False,False,False,False,False,False
311899,"Gîte n°443 ""LE GÎTE DU GABELOU""",2 épis,False,False,False,False,False,False,False,False,False
311939,"Restaurant ""La Principauté""",None,False,False,False,False,False,False,False,False,False
312075,"Gîte n°432 ""1843 LE RUISSEAU DU MOULIN""",3 épis / Confort,False,False,False,False,True,False,False,False,False
312172,"Restaurant ""Le Diapason""",Maître Restaurateur,False,False,False,True,False,False,False,False,False
312327,"Brasserie ""Seven Café""",None,False,False,False,False,False,False,False,False,False
312335,La Roseraie,None,False,False,False,False,False,False,False,False,False


### 4.4. Créer colonne is_etoile et colonne price_level

In [17]:
"""
Traitement de la colonne `rating_value` (DataTourisme)
Dans DataTourisme, la colonne `rating_value` correspond au nombre
d’étoiles officielles des hôtels (classement administratif),
et non à une note de satisfaction utilisateur.
Problèmes à adresser :
- Forte proportion de valeurs manquantes (~86 %)
- Variable numérique mais sémantiquement catégorielle
- Non comparable à des ratings type TripAdvisor ou Google
Objectif :
Transformer `rating_value` en deux variables métiers simples,
robustes et exploitables dans le scoring et l’UX :
1. `is_etoile` : indique si l’hôtel est classé (au moins 1 étoile)
2. `budget` : niveau de gamme estimé (eco / normal / premium)
"""

# -------------------------------------------------------------------
# 1. Création de la colonne `is_etoile`
# -------------------------------------------------------------------
# Logique métier :
# - True  → hôtel classé (rating_value >= 1)
# - False → non classé ou information manquante
# On utilise une approche vectorisée, robuste aux NaN.

df_clean["is_etoile"] = df_clean["rating_value"].ge(1).fillna(False)


# -------------------------------------------------------------------
# 2. Création de la colonne `budget` (3 niveaux)
# -------------------------------------------------------------------
# Hypothèses métier (simples et défendables) :
# - NaN, 1★, 2★ → "eco"      (non classé ou petit budget)
# - 3★          → "normal"   (milieu de gamme)
# - 4★, 5★      → "premium"  (haut de gamme)
# Cette variable ne représente PAS un niveau de satisfaction,
# mais un positionnement tarifaire / de gamme.

df_clean["price_level"] = np.select([df_clean["rating_value"].isna() | (df_clean["rating_value"] <= 2),
                                      df_clean["rating_value"] == 3,
                                      df_clean["rating_value"] >= 4],
                                     ["eco", "normal", "premium"],
                                     default="eco")

# -------------------------------------------------------------------
# Contôle de qualité
# -------------------------------------------------------------------

df_clean[["label_fr", "is_etoile", "price_level", "rating_value"]].head(10)

,label_fr,is_etoile,price_level,rating_value
312421,Le Monument aux Morts du Maquis des Manises et...,False,eco,NaN
312749,Sentier découverte du Vieil-Étang de Bairon,False,eco,NaN
311729,"Pizzéria ""Annabella""",False,eco,NaN
311896,Le Château de Charbogne,False,eco,NaN
311899,"Gîte n°443 ""LE GÎTE DU GABELOU""",True,eco,2.0
311939,"Restaurant ""La Principauté""",False,eco,NaN
312075,"Gîte n°432 ""1843 LE RUISSEAU DU MOULIN""",True,normal,3.0
312172,"Restaurant ""Le Diapason""",False,eco,NaN
312327,"Brasserie ""Seven Café""",False,eco,NaN
312335,La Roseraie,False,eco,NaN


### 4.5. Transformer Colonne poi_types en main_category

#### 4.5.1. Créer main_category et is_prime_plus & Calculer main_cat_weight

In [18]:
# ----------------------------------------------------------
# MAIN_CAT DES POIs (version optimisée + stable)
# ----------------------------------------------------------
"""
Objectif (Prime) :
- Chaque POI peut avoir plusieurs types (poi_types_clean = liste de labels nettoyés).
- Pour alimenter le scoring Prime, on veut une seule main_category par POI (déterministe).
- La stratégie est de convertir les types en catégories candidates, puis choisir la catégorie
  ayant le poids le plus élevé (CAT_WEIGHT).

Étapes du bloc :
1) CAT_WEIGHT : dictionnaire de pondération des main_category. Plus le poids est élevé, plus la catégorie
   est considérée "structurante" dans le scoring Prime. En cas de multi-type, la catégorie la plus lourde l’emporte.

2) TYPE_TO_CAT : Mapping fin entre chaque poi_type nettoyé (minuscule) et une main_category.
   Important : les clés doivent être exactement au format de poi_types_clean (lowercase),
   sinon les correspondances échoueront silencieusement.

3) pick_main_category_by_weight() :
Fonction cœur du choix de main_category pour un POI multi-type.

4) Application au DataFrame :
   - main_category : catégorie finale (celle retenue au poids max)
   - main_cat_weight : poids de la catégorie retenue (prêt pour le calcul Prime)
   - main_cat_candidates : liste des catégories candidates triées (debug/contrôle qualité)

Notes / bonnes pratiques :
- Ajuster IGNORE_TYPES si certains types sont trop génériques (ex: localbusiness) et polluent la catégorisation.
- Compléter TYPE_TO_CAT au fur et à mesure que de nouveaux types apparaissent.
- Contrôler la qualité en inspectant main_cat_candidates pour les POI ambigus (multi-catégories).
"""
#-------------------------------
# 1) Poids de tes main_category
#-------------------------------
CAT_WEIGHT = {
    "Culture & Musées": 0.9,
    "Patrimoine & Monuments": 0.8,
    "Nature & Paysages": 0.7,
    "Gastronomie & Restauration": 0.6,
    "Événements & Spectacles & Exposition" : 0.5,
    "Loisirs & Activités familiales": 0.4,
    "Shopping & Artisanat": 0.3,
    "Bien-être & Santé": 0.2,
    "Services & Pratique": 0.1,
    "Itinéraires & Circuits" : 0.0,
    "Hébergement": 0.0,}

#------------------------------------------------------------------
# 2) Mapping type -> main_category (mets tout en MINUSCULE, exactement comme poi_types_clean)
#------------------------------------------------------------------
TYPE_TO_CAT = {}

# Culture & Musées
for t in ["culturalsite"]:
    TYPE_TO_CAT[t] = "Culture & Musées"

# Patrimoine & Monuments
for t in ["archeologicalsite","abbey","basilica","cathedral","chapel","church","cloister","convent",
    "calvary","castle","citadel","bastide","aqueduct","bridge","collegiate","commanderie",
    "chartreuse","bishopric","cityheritage","house","civilcemetery","buddhisttemple"]:
    TYPE_TO_CAT[t] = "Patrimoine & Monuments"

# Gastronomie & Restauration
for t in ["foodestablishment","fastfoodrestaurant","cafeorcoffeeshop","bakery","coveredmarket"]:
    TYPE_TO_CAT[t] = "Gastronomie & Restauration"

# Hébergement
for t in ["accommodation","apartment"]:
    TYPE_TO_CAT[t] = "Hébergement"

# Événements & Spectacles & Exposition
for t in ["event","businessevent","circusplace", "auditorium"]:
    TYPE_TO_CAT[t] = "Événements & Spectacles & Exposition"

# Loisirs & Activités familiales
for t in ["library","cinematheque","educationaltrail",
          "amusementpark","adventurepark","bowlingalley","minigolf","golfcourse","climbingwall",
          "gymnasium","frontonbelotacourt","casino","movietheater","activityprovider","nauticalcentre",
          "marina","launchingramp","downhillskiresort","crosscountryskiresort","downhillskirun",
          "crosscountryskitrail","dogsleddingtrail", "aquarium", "product", "convenientservice", "civicstructure"]:
    TYPE_TO_CAT[t] = "Loisirs & Activités familiales"

# Nature & Paysages
for t in ["park","placeofinterest","landform", "arena"]:
    TYPE_TO_CAT[t] = "Nature & Paysages"

# Shopping & Artisanat
for t in ["localbusiness", "businessplace"]:
    TYPE_TO_CAT[t] = "Shopping & Artisanat"

# Bien-être & Santé
for t in ["balneotherapycentre","hammam"]:
    TYPE_TO_CAT[t] = "Bien-être & Santé"

# Services & Pratique
for t in ["touristinformationcenter", "airport","airfield","busstop","busstation", "equipmentrentalshop",
          "equipmentrepairshop", "multipurposeroomorcommunityroom"]:
    TYPE_TO_CAT[t] = "Services & Pratique"

# Itinéraires & Circuits
for t in ["orderedlist"]:
    TYPE_TO_CAT[t] = "Itinéraires & Circuits"

#------------------------------------------------------------------
# 3) Types à ignorer (si tu veux ignorer quelque chose, mets-le ici)
#------------------------------------------------------------------
IGNORE_TYPES = set()

def pick_main_category_by_weight(types_clean, type_to_cat=TYPE_TO_CAT, cat_weight=CAT_WEIGHT):
    """
    Fonction cœur du choix de main_category pour un POI multi-type.
   - Ignore les types "techniques" ou bruités via IGNORE_TYPES (ex: orderedlist).
   - Pour chaque type présent dans le POI :
       a) récupère la catégorie via TYPE_TO_CAT
       b) récupère son poids via CAT_WEIGHT
       c) alimente un dict "candidates" (cat -> poids) pour traçabilité
   - Sélectionne la catégorie avec le poids maximum (best_cat).
   - Retourne :
       - main_category retenue
       - son poids (main_cat_weight)
       - la liste triée des catégories candidates (main_cat_candidates)
   Fallback : si aucun type ne matche, retourne ("Autre", 0.0, []).
    types_clean: list[str] (ex: df_dt["poi_types_clean"])
    retourne: (main_category, main_weight, candidates_sorted)
    """
    if not isinstance(types_clean, (list, tuple, set)):
        return ("Autre", 0.0, [])

    best_cat = None
    best_w = -1.0

    # pour debug / traçabilité : toutes les catégories candidates avec leurs poids
    candidates = {}

    for t in types_clean:
        if not isinstance(t, str):
            continue
        t = t.strip().lower()
        if t in IGNORE_TYPES:
            continue

        cat = type_to_cat.get(t)
        if not cat:
            continue

        w = cat_weight.get(cat, 0.0)
        # garder le max par catégorie (au cas où plusieurs types mènent à la même cat)
        candidates[cat] = max(candidates.get(cat, 0.0), w)

        if w > best_w:
            best_w = w
            best_cat = cat

    if best_cat is None:
        return ("Autre", 0.0, [])

    candidates_sorted = sorted(candidates.items(), key=lambda x: x[1], reverse=True)
    return (best_cat, best_w, candidates_sorted)


# 4) Application dataframe (version simple)
df_clean["main_category"] = df_clean["poi_types_clean"].apply(lambda xs: pick_main_category_by_weight(xs)[0])

#flag Itinéraires & Circuits = colonne "is_prime_plus"
df_clean["is_prime_plus"] = df_clean["poi_types_clean"].apply(lambda xs: isinstance(xs, (list, tuple, set)) and ("orderedlist" in xs))

# (optionnel) garder le poids retenu + les candidats pour expliquer/contrôler
df_clean["main_cat_weight"] = df_clean["poi_types_clean"].apply(lambda xs: pick_main_category_by_weight(xs)[1])
df_clean["main_cat_candidates"] = df_clean["poi_types_clean"].apply(lambda xs: pick_main_category_by_weight(xs)[2])


# ----------------------------------------------------------
# CONTRÔLE QUALITÉ
# ----------------------------------------------------------
df_clean[df_clean["poi_types_clean"].apply(lambda x: isinstance(x, list) and "product" in x)][
    ["poi_types_clean", "main_category", "main_cat_weight","label_fr", "review_compliant_with", "is_prime_plus", "is_resto", "contact_homepage"]].head(10)

# contrôle qualité
#mask = (df_clean["poi_types_clean"].apply(lambda x: isinstance(x, list) and "culturalsite" in x) & df_clean["review_compliant_with"].eq("CulturalSite"))
#df_culturalsite_review = df_clean[mask]
#df_culturalsite_review[["poi_types_clean", "main_category", "main_cat_weight", "label_fr", "review_compliant_with", "contact_homepage"]].head(50)

,poi_types_clean,main_category,main_cat_weight,label_fr,review_compliant_with,is_prime_plus,is_resto,contact_homepage
312464,[product],Loisirs & Activités familiales,0.4,Domaine Château du Risdoux : Location vélos,PlaceOfInterest,False,False,https://www.chateaulerisdoux.fr/
313155,[product],Loisirs & Activités familiales,0.4,Base de Loisirs du Domaine de Vendresse,None,False,False,http://www.domaine-de-vendresse.fr/
312027,[product],Loisirs & Activités familiales,0.4,Horloge du Grand Marionnettiste,None,False,False,http://www.charleville-mezieres.fr/Culture-pat...
313489,[product],Loisirs & Activités familiales,0.4,Location de canoés,None,False,False,https://terrassedes4filsaymon.fr/
314026,[product],Loisirs & Activités familiales,0.4,Ferme pédagogique de Liart,CampingAndCaravanning,False,False,https://www.fermepedagogiqueliart.fr/
311803,[product],Loisirs & Activités familiales,0.4,Arden'Gyropode,None,False,False,http://www.robinson-ardennes.com/
313483,[product],Loisirs & Activités familiales,0.4,Location VTT -Hôtel Val Saint-Hilaire,None,False,False,None
277886,[product],Loisirs & Activités familiales,0.4,VIVEZ LA MAGIE DES SAISONS EN PLEINE NATURE !,None,False,False,None
280806,[product],Loisirs & Activités familiales,0.4,VISITE GUIDÉE,None,False,False,None
259861,[product],Loisirs & Activités familiales,0.4,Visite nocturne de la chapelle des Pénitents N...,None,False,False,https://boutique.bastides-gorges-aveyron.fr/vi...


#### 4.5.2. Créer format_label, format_weight

In [19]:
# ----------------------------------------------------------
# FORMAT DES POIs (bugfix cache + stable)
# ----------------------------------------------------------
import ast
from collections import Counter

"""
FORMAT DES POIs
Le "format" décrit la nature de l'interaction utilisateur avec un POI, indépendamment de sa catégorie thématique.
- lieu        : POI visitable / contemplatif (sightseeing)
- expérience  : POI impliquant une activité ou une action
- besoin      : POI utilitaire ou fonctionnel (restaurant, hébergement)
- parcours    : objet macro (itinéraires longs, hors Prime, réservé pour Prime+)
"""

# ----------------------------------------------------------
# 1) MAPPING : type -> format
# ----------------------------------------------------------
TYPE_TO_FORMAT = {}

for t in [
    "culturalsite", "archeologicalsite", "abbey", "basilica", "cathedral", "chapel", "church",
    "cloister", "convent", "calvary", "castle", "citadel", "bastide", "aqueduct", "bridge",
    "collegiate", "commanderie", "chartreuse", "bishopric", "cityheritage", "house",
    "civilcemetery", "buddhisttemple", "park", "placeofinterest", "landform"
]:
    TYPE_TO_FORMAT[t] = "lieu"

for t in [
    "event", "businessevent", "arena", "circusplace", "library", "cinematheque", "auditorium", "educationaltrail",
    "amusementpark", "adventurepark", "bowlingalley", "minigolf", "golfcourse", "climbingwall",
    "gymnasium", "frontonbelotacourt", "casino", "movietheater", "activityprovider", "nauticalcentre",
    "marina", "launchingramp", "downhillskiresort", "crosscountryskiresort", "downhillskirun",
    "crosscountryskitrail", "dogsleddingtrail", "aquarium", "businessplace",
    "localbusiness", "equipmentrentalshop", "equipmentrepairshop"
]:
    TYPE_TO_FORMAT[t] = "expérience"

for t in [
    "balneotherapycentre", "hammam", "touristinformationcenter", "convenientservice", "civicstructure",
    "airport", "airfield", "busstop", "busstation", "accommodation", "apartment", "multipurposeroomorcommunityroom",
    "foodestablishment", "fastfoodrestaurant", "cafeorcoffeeshop", "bakery", "coveredmarket", "product"
]:
    TYPE_TO_FORMAT[t] = "besoin"

TYPE_TO_FORMAT["orderedlist"] = "parcours"

FORMAT_WEIGHT = {"lieu": 0.09, "expérience": 0.06, "besoin": 0.03, "parcours": 0.0}

# ----------------------------------------------------------
# 2) NORMALISATION
# ----------------------------------------------------------
def _as_list(x):
    """
    Convertit poi_types en liste de strings normalisées (strip + lower).
    """
    if isinstance(x, list):
        return [str(v).strip().lower() for v in x if str(v).strip()]

    if isinstance(x, str):
        s = x.strip()

        # string représentant une liste Python
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                vals = list(ast.literal_eval(s))
                return [str(v).strip().lower() for v in vals if str(v).strip()]
            except Exception:
                pass

        # style "park,culturalsite"
        if "," in s:
            return [v.strip().lower() for v in s.split(",") if v.strip()]

        return [s.lower()] if s else []

    return []

# ----------------------------------------------------------
# 3) CALCUL format_label
# ----------------------------------------------------------
def compute_format_label(poi_types):
    """
    Calcule le format dominant à partir des types.
    """
    formats = []
    for t in _as_list(poi_types):
        f = TYPE_TO_FORMAT.get(t)
        if f:
            formats.append(f)

    if not formats:
        return "lieu"  # fallback

    counts = Counter(formats)
    top_n = max(counts.values())
    top_formats = [k for k, v in counts.items() if v == top_n]

    priority = {"lieu": 3, "expérience": 2, "besoin": 1, "parcours": 0}
    top_formats.sort(key=lambda k: priority.get(k, 0), reverse=True)
    return top_formats[0]

# ----------------------------------------------------------
# 4) CACHE (BUGFIX ICI)
# ----------------------------------------------------------
_format_cache = {}

def compute_format_label_cached(poi_types):
    """
    Cache : la clé est un tuple de types normalisés.
    BUGFIX : on calcule la valeur avec poi_types ORIGINAL (pas avec la clé tuple).
    """
    key = tuple(_as_list(poi_types))          # clé de cache (tuple)
    if key in _format_cache:
        return _format_cache[key]

    val = compute_format_label(poi_types)    # ✅ BUGFIX : on passe poi_types, pas "key"
    _format_cache[key] = val
    return val

# ----------------------------------------------------------
# 5) APPLICATION AU DF + QC
# ----------------------------------------------------------
_format_cache = {}  # reset avant recalcul

df_clean["format_label"] = df_clean["poi_types_clean"].apply(compute_format_label_cached)
df_clean["format_weight"] = df_clean["format_label"].map(FORMAT_WEIGHT).fillna(0.0)

print("Distribution format_label :")
print(df_clean["format_label"].value_counts(dropna=False).head(10))

print("\nExemples 'besoin' (accommodation / foodestablishment) :")
mask_besoin = df_clean["poi_types_clean"].apply(lambda x: any(t in {"accommodation", "foodestablishment"} for t in _as_list(x)))
display(df_clean.loc[mask_besoin, ["poi_types_clean", "main_category", "format_label", "format_weight", "label_fr"]].head(10))

print("\nExemples 'expérience' (event) :")
mask_exp = df_clean["poi_types_clean"].apply(lambda x: "event" in _as_list(x))
display(df_clean.loc[mask_exp, ["poi_types_clean", "main_category", "format_label", "format_weight", "label_fr"]].head(10))

print("\nExemples 'lieu' :")
mask_exp = df_clean["poi_types_clean"].apply(lambda x: "culturalsite" in _as_list(x))
display(df_clean.loc[mask_exp, ["poi_types_clean", "main_category", "format_label", "format_weight", "label_fr"]].head(10))

Distribution format_label :
format_label
besoin        190091
expérience    134224
lieu           65336
parcours       21983
Name: count, dtype: int64

Exemples 'besoin' (accommodation / foodestablishment) :


,poi_types_clean,main_category,format_label,format_weight,label_fr
311729,[foodestablishment],Gastronomie & Restauration,besoin,0.03,"Pizzéria ""Annabella"""
311896,[accommodation],Hébergement,besoin,0.03,Le Château de Charbogne
311899,[accommodation],Hébergement,besoin,0.03,"Gîte n°443 ""LE GÎTE DU GABELOU"""
311939,[accommodation],Hébergement,besoin,0.03,"Restaurant ""La Principauté"""
312075,[accommodation],Hébergement,besoin,0.03,"Gîte n°432 ""1843 LE RUISSEAU DU MOULIN"""
312172,[foodestablishment],Gastronomie & Restauration,besoin,0.03,"Restaurant ""Le Diapason"""
312327,[foodestablishment],Gastronomie & Restauration,besoin,0.03,"Brasserie ""Seven Café"""
312335,[accommodation],Hébergement,besoin,0.03,La Roseraie
312557,[foodestablishment],Gastronomie & Restauration,besoin,0.03,"Restaurant ""Chez Toshi"""
312874,[accommodation],Hébergement,besoin,0.03,"Hôtel ""Inn Design"""



Exemples 'expérience' (event) :


,poi_types_clean,main_category,format_label,format_weight,label_fr
314302,[event],Événements & Spectacles & Exposition,expérience,0.06,Soirée sport & détente
313698,[event],Événements & Spectacles & Exposition,expérience,0.06,"Théâtre conte ""Pour que tu m'aimes encore"""
313857,[event],Événements & Spectacles & Exposition,expérience,0.06,Blind test : Jeux vidéos & animés
275999,[event],Événements & Spectacles & Exposition,expérience,0.06,CHANDELEUR À LARRA
277020,[event],Événements & Spectacles & Exposition,expérience,0.06,"DÎNER-SPECTACLE ""14 JUILLET À LA MAISON DE RE..."
277446,[event],Événements & Spectacles & Exposition,expérience,0.06,DUO BERNOT-LUCIANI AMÉRIQUES
278405,[event],Événements & Spectacles & Exposition,expérience,0.06,"SPECTACLE ENFANT ""MINUS"""
278765,[event],Événements & Spectacles & Exposition,expérience,0.06,"SPECTACLE ENFANT ""DANS LE NID DE LA SOURIS"""
279410,[event],Événements & Spectacles & Exposition,expérience,0.06,MOIS DU POLAR À MONTAIGUT ET ST PAUL !
281331,[event],Événements & Spectacles & Exposition,expérience,0.06,"SPECTACLE ENFANT ""BOUGE"""



Exemples 'lieu' :


,poi_types_clean,main_category,format_label,format_weight,label_fr
312421,[culturalsite],Culture & Musées,lieu,0.09,Le Monument aux Morts du Maquis des Manises et...
313643,[culturalsite],Culture & Musées,lieu,0.09,Ancien Relais de Poste et de Messageries
312333,[culturalsite],Culture & Musées,lieu,0.09,Ancien lavoir
312058,[culturalsite],Culture & Musées,lieu,0.09,Pressoir de Margy
312398,[culturalsite],Culture & Musées,lieu,0.09,Ancienne Poudrerie Royale de Saint-Ponce
313911,[culturalsite],Culture & Musées,lieu,0.09,Stèle de Roland Garros
314150,[culturalsite],Culture & Musées,lieu,0.09,Ancienne cité ouvrière
312041,[culturalsite],Culture & Musées,lieu,0.09,Ancienne usine métallurgique dite la Forge Gen...
312574,[culturalsite],Culture & Musées,lieu,0.09,Ancienne usine de décolletage
276796,[culturalsite],Culture & Musées,lieu,0.09,SALLE D'EXPOSITION DU FOYER


#### 4.5.3. Créer tempo_label, tempo_weight

In [20]:
# ============================================================
# TEMPO DES POIs (version corrigée + stable + notebook-proof)
# ============================================================
from collections import Counter

"""
TEMPO DES POIs
Le "tempo" décrit le rythme de visite induit par un POI dans une journée.
- lent       : POI long / immersif / contemplatif
- normal     : POI standard
- dynamique  : POI rapide / fluide / de passage
- zen        : itinéraires (macro-objets)
Le tempo est utilisé dans le scoring Prime pour moduler la densité d’un itinéraire journalier.
"""

# ============================================================
# MAPPING : type -> tempo
# ============================================================
TYPE_TO_TEMPO = {}

for t in [
    "culturalsite", "archeologicalsite", "abbey", "basilica", "cathedral", "chapel", "church",
    "cloister", "convent", "calvary", "castle", "citadel", "bastide", "aqueduct", "bridge",
    "collegiate", "commanderie", "chartreuse", "bishopric", "cityheritage", "house",
    "civilcemetery", "buddhisttemple", "park", "placeofinterest", "landform"
]:
    TYPE_TO_TEMPO[t] = "lent"

for t in [
    "balneotherapycentre", "hammam", "touristinformationcenter", "convenientservice", "civicstructure",
    "airport", "airfield", "busstop", "busstation", "accommodation", "apartment", "multipurposeroomorcommunityroom",
    "foodestablishment", "fastfoodrestaurant", "cafeorcoffeeshop", "bakery", "coveredmarket", "product"
]:
    TYPE_TO_TEMPO[t] = "normal"

for t in [
    "event", "businessevent", "arena", "circusplace", "library", "cinematheque", "auditorium", "educationaltrail",
    "amusementpark", "adventurepark", "bowlingalley", "minigolf", "golfcourse", "climbingwall",
    "gymnasium", "frontonbelotacourt", "casino", "movietheater", "activityprovider", "nauticalcentre",
    "marina", "launchingramp", "downhillskiresort", "crosscountryskiresort", "downhillskirun",
    "crosscountryskitrail", "dogsleddingtrail", "aquarium", "businessplace",
    "localbusiness", "equipmentrentalshop", "equipmentrepairshop"
]:
    TYPE_TO_TEMPO[t] = "dynamique"

TYPE_TO_TEMPO["orderedlist"] = "zen"

TEMPO_WEIGHT = {
    "lent": 0.09,
    "normal": 0.06,
    "dynamique": 0.03,
    "zen": 0.0
}

# ============================================================
# CALCUL DU TEMPO DOMINANT PAR POI
# ============================================================
def compute_tempo_label(poi_types):
    tempos = []
    for t in _as_list(poi_types):
        tempo = TYPE_TO_TEMPO.get(t)
        if tempo:
            tempos.append(tempo)

    if not tempos:
        return "normal"  # fallback

    counts = Counter(tempos)
    top_n = max(counts.values())
    top_tempos = [k for k, v in counts.items() if v == top_n]

    priority = {"lent": 3, "normal": 2, "dynamique": 1, "zen": 0}
    top_tempos.sort(key=lambda k: priority.get(k, 0), reverse=True)
    return top_tempos[0]

# ============================================================
# APPLICATION AU DATAFRAME (cache corrigé)
# ============================================================
_tempo_cache = {}  # reset

def compute_tempo_label_cached(poi_types):
    """
    Cache : clé = tuple(types normalisés)
    BUGFIX : on calcule avec poi_types ORIGINAL (pas avec la clé tuple)
    """
    key = tuple(_as_list(poi_types))
    if key in _tempo_cache:
        return _tempo_cache[key]

    val = compute_tempo_label(poi_types)
    _tempo_cache[key] = val
    return val

df_clean["tempo_label"] = df_clean["poi_types_clean"].apply(compute_tempo_label_cached)
df_clean["tempo_weight"] = df_clean["tempo_label"].map(TEMPO_WEIGHT).fillna(0.6)

# ============================================================
# CONTRÔLE QUALITÉ (tests simples et parlants)
# ============================================================
print(df_clean["tempo_label"].value_counts(dropna=False).head(10))

print("\nExemples 'lent' attendus (culturalsite) :")
mask_lent = df_clean["poi_types_clean"].apply(lambda x: "culturalsite" in _as_list(x))
display(df_clean.loc[mask_lent, ["poi_types_clean","tempo_label","tempo_weight","label_fr"]].head(10))

print("\nExemples 'dynamique' attendus (event) :")
mask_dyn = df_clean["poi_types_clean"].apply(lambda x: "event" in _as_list(x))
display(df_clean.loc[mask_dyn, ["poi_types_clean","tempo_label","tempo_weight","label_fr"]].head(10))

print("\nExemples 'normal' attendus (accommodation / foodestablishment) :")
mask_norm = df_clean["poi_types_clean"].apply(lambda x: any(t in {"accommodation","foodestablishment"} for t in _as_list(x)))
display(df_clean.loc[mask_norm, ["poi_types_clean","tempo_label","tempo_weight","label_fr"]].head(10))

tempo_label
normal       190429
dynamique    134224
lent          64998
zen           21983
Name: count, dtype: int64

Exemples 'lent' attendus (culturalsite) :


,poi_types_clean,tempo_label,tempo_weight,label_fr
312421,[culturalsite],lent,0.09,Le Monument aux Morts du Maquis des Manises et...
313643,[culturalsite],lent,0.09,Ancien Relais de Poste et de Messageries
312333,[culturalsite],lent,0.09,Ancien lavoir
312058,[culturalsite],lent,0.09,Pressoir de Margy
312398,[culturalsite],lent,0.09,Ancienne Poudrerie Royale de Saint-Ponce
313911,[culturalsite],lent,0.09,Stèle de Roland Garros
314150,[culturalsite],lent,0.09,Ancienne cité ouvrière
312041,[culturalsite],lent,0.09,Ancienne usine métallurgique dite la Forge Gen...
312574,[culturalsite],lent,0.09,Ancienne usine de décolletage
276796,[culturalsite],lent,0.09,SALLE D'EXPOSITION DU FOYER



Exemples 'dynamique' attendus (event) :


,poi_types_clean,tempo_label,tempo_weight,label_fr
314302,[event],dynamique,0.03,Soirée sport & détente
313698,[event],dynamique,0.03,"Théâtre conte ""Pour que tu m'aimes encore"""
313857,[event],dynamique,0.03,Blind test : Jeux vidéos & animés
275999,[event],dynamique,0.03,CHANDELEUR À LARRA
277020,[event],dynamique,0.03,"DÎNER-SPECTACLE ""14 JUILLET À LA MAISON DE RE..."
277446,[event],dynamique,0.03,DUO BERNOT-LUCIANI AMÉRIQUES
278405,[event],dynamique,0.03,"SPECTACLE ENFANT ""MINUS"""
278765,[event],dynamique,0.03,"SPECTACLE ENFANT ""DANS LE NID DE LA SOURIS"""
279410,[event],dynamique,0.03,MOIS DU POLAR À MONTAIGUT ET ST PAUL !
281331,[event],dynamique,0.03,"SPECTACLE ENFANT ""BOUGE"""



Exemples 'normal' attendus (accommodation / foodestablishment) :


,poi_types_clean,tempo_label,tempo_weight,label_fr
311729,[foodestablishment],normal,0.06,"Pizzéria ""Annabella"""
311896,[accommodation],normal,0.06,Le Château de Charbogne
311899,[accommodation],normal,0.06,"Gîte n°443 ""LE GÎTE DU GABELOU"""
311939,[accommodation],normal,0.06,"Restaurant ""La Principauté"""
312075,[accommodation],normal,0.06,"Gîte n°432 ""1843 LE RUISSEAU DU MOULIN"""
312172,[foodestablishment],normal,0.06,"Restaurant ""Le Diapason"""
312327,[foodestablishment],normal,0.06,"Brasserie ""Seven Café"""
312335,[accommodation],normal,0.06,La Roseraie
312557,[foodestablishment],normal,0.06,"Restaurant ""Chez Toshi"""
312874,[accommodation],normal,0.06,"Hôtel ""Inn Design"""


#### 4.5.4. Calculer score_prime

In [21]:
# ============================================================
# CALCUL DU SCORE FINAL PRIME
# ============================================================
"""
Le score Prime final repose sur la catégorie principale comme facteur dominant.
Le format et le tempo sont des modulateurs légers qui ajustent :
- la nature de l’expérience (format)
- le rythme de la journée (tempo)
Formule :
FINAL_SCORE = main_cat_weight * (1 + format_weight + tempo_weight)
Aucune pondération liée aux labels (incontournable, green, etc.)
n’intervient ici afin de préserver la neutralité du moteur Prime.
"""

# Sécurisation des valeurs manquantes
df_clean["main_cat_weight"] = df_clean["main_cat_weight"].fillna(0.0)
df_clean["format_weight"] = df_clean["format_weight"].fillna(0.0)
df_clean["tempo_weight"] = df_clean["tempo_weight"].fillna(0.0)

# Calcul vectorisé (rapide et stable)
df_clean["score_prime"] = (df_clean["main_cat_weight"] * (1 + df_clean["format_weight"] + df_clean["tempo_weight"]))

# ============================================================
# CONTRÔLE QUALITÉ (échantillon)
# ============================================================
df_clean[df_clean["score_prime"] > 0][["label_fr", "main_category",
                                       "main_cat_weight", "format_label", "format_weight",
                                       "tempo_label", "tempo_weight", "score_prime",]].head(20)

,label_fr,main_category,main_cat_weight,format_label,format_weight,tempo_label,tempo_weight,score_prime
312421,Le Monument aux Morts du Maquis des Manises et...,Culture & Musées,0.9,lieu,0.09,lent,0.09,1.062
311729,"Pizzéria ""Annabella""",Gastronomie & Restauration,0.6,besoin,0.03,normal,0.06,0.654
312172,"Restaurant ""Le Diapason""",Gastronomie & Restauration,0.6,besoin,0.03,normal,0.06,0.654
312327,"Brasserie ""Seven Café""",Gastronomie & Restauration,0.6,besoin,0.03,normal,0.06,0.654
312385,Aire d'accueil et de services pour cyclo-touri...,Nature & Paysages,0.7,lieu,0.09,lent,0.09,0.826
312464,Domaine Château du Risdoux : Location vélos,Loisirs & Activités familiales,0.4,besoin,0.03,normal,0.06,0.436
312557,"Restaurant ""Chez Toshi""",Gastronomie & Restauration,0.6,besoin,0.03,normal,0.06,0.654
312910,"Brasserie ""Le Cardinal""",Gastronomie & Restauration,0.6,besoin,0.03,normal,0.06,0.654
313389,Vannerie d'Ardenne,Shopping & Artisanat,0.3,expérience,0.06,dynamique,0.03,0.327
313411,L'Odyssée,Gastronomie & Restauration,0.6,besoin,0.03,normal,0.06,0.654


# 5. DataFrame Final

### 5.1. Normalisation les colonnes

In [43]:
"""
Normalisation des colonnes
- Renommage orienté produit (UI-first).
- Typage et downcast systématiques (float32, Int*, bool) afin de réduire l’empreinte mémoire et la taille des fichiers parquet.
- Nettoyage minimal (lat / lon requis).
- Conversion des identifiants et codes (ex: postal_code) en formats stables et cohérents pour l’affichage.
"""
def safe_str(s: pd.Series) -> pd.Series:
    """String propre : strip, NA conservés"""
    return (
        s.astype("string")
         .str.strip()
         .replace("", pd.NA))

def to_float32(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype("float32")

def to_int32_nullable(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype("Int32")

def to_int16_nullable(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype("Int16")

def list_to_str(s: pd.Series) -> pd.Series:
    """liste → 'a|b|c' ; NA si pas liste"""
    return s.apply(
        lambda x: "|".join(map(str, x)) if isinstance(x, list) and len(x) > 0 else pd.NA
    ).astype("string")

def normalize_datatourisme(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise l'ensemble des colonnes DataTourisme :
    - types cohérents
    - strings nettoyées
    - floats downcastés
    - identifiants stables
    Aucun filtrage, aucun split.
    """
    df = df.copy()

    # --------------------
    # Identifiants / labels
    # --------------------
    df["poi_id"] = df["poi_id"].astype("string")
    df["label_fr"] = safe_str(df["label_fr"])
    df["label_en"] = safe_str(df["label_en"])

    # --------------------
    # Descriptions
    # --------------------
    df["short_desc_fr"] = safe_str(df["short_desc_fr"])
    df["short_desc_en"] = safe_str(df["short_desc_en"])

    # --------------------
    # Géolocalisation
    # --------------------
    df["latitude"] = to_float32(df["latitude"])
    df["longitude"] = to_float32(df["longitude"])

    # --------------------
    # Localisation administrative
    # --------------------
    df["country_label_fr"] = safe_str(df["country_label_fr"])
    df["region_label_fr"] = safe_str(df["region_label_fr"])
    df["dept_label_fr"] = safe_str(df["dept_label_fr"])
    df["city_label_fr"] = safe_str(df["city_label_fr"])

    df["region_insee"] = to_int32_nullable(df["region_insee"])
    df["dept_insee"] = to_int16_nullable(df["dept_insee"])
    df["city_insee"] = to_int32_nullable(df["city_insee"])

    # postal_code : identifiant → Int32 (pas float)
    df["postal_code"] = to_int32_nullable(df["postal_code"])

    # --------------------
    # Adresse / contact
    # --------------------
    df["address_locality"] = safe_str(df["address_locality"])
    df["street_address"] = safe_str(df["street_address"])
    df["contact_homepage"] = safe_str(df["contact_homepage"])

    # --------------------
    # Médias
    # --------------------
    df["main_media_url"] = safe_str(df["main_media_url"])
    df["media_resource_url"] = safe_str(df["media_resource_url"])

    # --------------------
    # Rating / capacité
    # --------------------
    df["rating_value"] = to_float32(df["rating_value"])
    df["allowed_persons"] = to_int16_nullable(df["allowed_persons"])

    # --------------------
    # Prix
    # --------------------
    df["price"] = to_float32(df["price"])
    df["min_price"] = to_float32(df["min_price"])
    df["max_price"] = to_float32(df["max_price"])
    df["price_level"] = safe_str(df["price_level"])

    # --------------------
    # Catégorisation
    # --------------------
    df["main_category"] = safe_str(df["main_category"])
    df["format_label"] = safe_str(df["format_label"])
    df["tempo_label"] = safe_str(df["tempo_label"])

    df["main_cat_weight"] = to_float32(df["main_cat_weight"])
    df["format_weight"] = to_float32(df["format_weight"])
    df["tempo_weight"] = to_float32(df["tempo_weight"])
    df["score_prime"] = to_float32(df["score_prime"])

    df["main_cat_candidates"] = list_to_str(df["main_cat_candidates"])

    # --------------------
    # Booléens (déjà OK, on force juste)
    # --------------------
    BOOL_COLS = [
        "is_resto_type", "is_resto_label", "is_resto",
        "is_label_incontournable", "is_label_famille", "is_label_handicap",
        "is_label_hebergement", "is_label_gastronomie", "is_label_artisanat",
        "is_label_randonnee", "is_label_green",
        "is_etoile", "is_prime_plus",
    ]
    for c in BOOL_COLS:
        df[c] = df[c].astype(bool)

    # --------------------
    # Colonnes excursion (normalisées mais pas utilisées encore)
    # --------------------
    df["difficulty_level_fr"] = safe_str(df["difficulty_level_fr"])
    df["locomotion_mode_fr"] = safe_str(df["locomotion_mode_fr"])
    df["tour_type_fr"] = safe_str(df["tour_type_fr"])

    df["tour_distance_m"] = to_float32(df["tour_distance_m"])
    df["duration_min"] = to_float32(df["duration_min"])
    df["practice_duration_min"] = to_float32(df["practice_duration_min"])
    df["duration_days"] = to_float32(df["duration_days"])
    df["practice_duration_days"] = to_float32(df["practice_duration_days"])
    df["positive_elevation_gain_m"] = to_float32(df["positive_elevation_gain_m"])
    df["negative_elevation_loss_m"] = to_float32(df["negative_elevation_loss_m"])

    df["start_date"] = safe_str(df["start_date"])
    df["end_date"] = safe_str(df["end_date"])
    df["hours_valid_from"] = safe_str(df["hours_valid_from"])
    df["hours_valid_through"] = safe_str(df["hours_valid_through"])

    # --------------------
    # Métadonnées
    # --------------------
    df["last_update_datatourisme"] = safe_str(df["last_update_datatourisme"])

    return df

df_norm = normalize_datatourisme(df_clean)
df_norm.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
Index: 411634 entries, 312421 to 290134
Data columns (total 73 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   poi_id                     411634 non-null  string 
 1   poi_types                  411634 non-null  object 
 2   label_fr                   411634 non-null  string 
 3   label_en                   184799 non-null  string 
 4   short_desc_fr              262567 non-null  string 
 5   short_desc_en              263911 non-null  string 
 6   latitude                   411634 non-null  float32
 7   longitude                  411634 non-null  float32
 8   country_label_fr           411634 non-null  string 
 9   region_insee               411634 non-null  Int32  
 10  region_label_fr            411634 non-null  string 
 11  dept_insee                 410978 non-null  Int16  
 12  dept_label_fr              411634 non-null  string 
 13  city_insee                 41

### 5.2. Renommage + Split le dataframe en 2 : prim_classique & prime_excursion

In [52]:
"""
OBJECTIF DU BLOC

Ce bloc prépare le dataset de base destiné à l’interface utilisateur
de l’application (Streamlit), à partir du dataframe DataTourisme
préalablement normalisé (`df_norm`).

1) Construction d’un dataframe UI unifié
   - Sélection et renommage explicites des colonnes nécessaires à l’affichage,
     au filtrage et aux calculs futurs (distance, scoring).
   - Ajout de colonnes fonctionnelles absentes de la source :
       • `source_id` et `source` pour l’identification et la traçabilité
       • `review_count` et `distance_km` comme placeholders produits
   - Normalisation des formats pour l’UX (strings propres, types numériques stables).
   - Aucune logique métier Prime / Excursion n’est encore appliquée à ce stade.

2) Split fonctionnel du dataset par lignes
   - Le dataframe UI est séparé en deux sous-ensembles selon la catégorie principale :
       • Prime classique :
         toutes les lignes dont `main_category_compressed` ≠ "Itinéraires & Circuits"
       • Prime excursion :
         uniquement les lignes dont `main_category_compressed` = "Itinéraires & Circuits"
   - Ce split par lignes permet de distinguer clairement les usages
     sans dupliquer les calculs ni les transformations.

3) Enregistrement des dataframes en parquet compressé
"""

# ============================================================
# 1) Construire le dataframe UI (renommage + colonnes nécessaires)
# ============================================================
RENAME_MAP = {
    # ids / source
    "poi_id": "source_id",

    # géoloc
    "latitude": "lat",
    "longitude": "lon",

    # contenu
    "label_fr": "name",
    "contact_homepage": "url",

    # adresse
    "street_address": "address",
    "city_label_fr": "city",
    "region_label_fr": "region",
    "dept_label_fr": "departement",
    "country_label_fr": "country",

    # catégories
    "main_category": "main_category_compressed",
    "review_compliant_with": "sub_category",
    "poi_types_clean": "type_principal",
    "short_desc_fr" : "snippet",
    "short_desc_en" : "snippet_en",
    "rating_value": "rating",
    "allowed_persons": "max_people",   
}

df_ui = df_norm.rename(columns=RENAME_MAP).copy()

df_ui["source"] = "datatourisme"
df_ui["review_count"] = pd.Series(pd.NA, index=df_ui.index, dtype="Int32")
df_ui["distance_km"] = pd.Series(np.nan, index=df_ui.index, dtype="float32")


# ============================================================
# 2) Splitter en 2 dataframes (Prime classique vs Excursion)
# ============================================================
EXCURSION_CAT = "Itinéraires & Circuits"
df_prime_classique = df_ui[df_ui["main_category_compressed"] != EXCURSION_CAT].copy()
df_prime_excursion = df_ui[df_ui["main_category_compressed"] == EXCURSION_CAT].copy()


# ============================================================
# 3) Enregistrement des dataframes en parquet compressé
# ============================================================

out_prime_classique = "C:\\Users\\DELL\\Downloads\\ItineraireVacances3\\df_prime_classique.parquet"
out_prime_excursion = "C:\\Users\\DELL\\Downloads\\ItineraireVacances3\\df_prime_excursion.parquet"

def save_parquet_safe(df: pd.DataFrame, path: str):
    """
    Enregistre un dataframe en parquet compressé.
    Priorité à zstd (niveau 10), fallback automatique vers snappy.
    """
    try:
        df.to_parquet(
            path,
            index=False,
            compression="zstd",
            compression_level=10)
        print("saved with zstd level 10:", path)
    except TypeError:
        # compression_level non supporté
        df.to_parquet(path, index=False, compression="zstd")
        print("saved with zstd (default level):", path)
    except Exception:
        # fallback ultime
        df.to_parquet(path, index=False, compression="snappy")
        print("saved with snappy:", path)


# Sauvegarde
save_parquet_safe(df_prime_classique, out_prime_classique)
save_parquet_safe(df_prime_excursion, out_prime_excursion)
out_prime_classique, out_prime_excursion

# Contrôle qualité
print(f"UI            : {df_ui.shape[0]} lignes × {df_ui.shape[1]} colonnes")
print(f"Prime classique: {df_prime_classique.shape[0]} lignes × {df_prime_classique.shape[1]} colonnes")
print(f"Prime excursion: {df_prime_excursion.shape[0]} lignes × {df_prime_excursion.shape[1]} colonnes")

saved with zstd level 10: C:\Users\DELL\Downloads\ItineraireVacances3\df_prime_classique.parquet
saved with zstd level 10: C:\Users\DELL\Downloads\ItineraireVacances3\df_prime_excursion.parquet
UI            : 411634 lignes × 76 colonnes
Prime classique: 389651 lignes × 76 colonnes
Prime excursion: 21983 lignes × 76 colonnes


### 5.3. Contrôle de qualité

In [53]:
display(df_prime_classique.head(3))
display(df_prime_classique.info())

,source_id,poi_types,name,label_en,snippet,snippet_en,lat,lon,country,region_insee,region,dept_insee,departement,city_insee,city,postal_code,address_locality,address,theme_fr,theme_en,architectural_style_fr,architectural_style_en,main_media_url,media_resource_url,url,rating,review_value_label_fr,review_value_label_en,sub_category,opens_time,closes_time,hours_valid_from,hours_valid_through,max_people,difficulty_level_fr,locomotion_mode_fr,tour_type_fr,tour_distance_m,duration_min,practice_duration_min,duration_days,practice_duration_days,positive_elevation_gain_m,negative_elevation_loss_m,start_date,end_date,last_update_datatourisme,price,min_price,max_price,type_principal,is_resto_type,is_resto_label,is_resto,is_label_incontournable,is_label_famille,is_label_handicap,is_label_hebergement,is_label_gastronomie,is_label_artisanat,is_label_randonnee,is_label_green,is_etoile,price_level,main_category_compressed,is_prime_plus,main_cat_weight,main_cat_candidates,format_label,format_weight,tempo_label,tempo_weight,score_prime,source,review_count,distance_km
312421,https://data.datatourisme.fr/42/462e6825-56eb-...,CulturalSite,Le Monument aux Morts du Maquis des Manises et...,<NA>,"Le 13 juin 1944, alors que la deuxième guerre ...","On 13 June 1944, as the Second World War enter...",49.942829,4.649277,France,44,Grand Est,8,Ardennes,8363,Revin,8500,Revin,Route du Mont Malgré Tout,Histoire,History,None,None,http://cnstlltn.com/1024x768/4d80126f-fd33-439...,http://cnstlltn.com/1024x768/088c3fff-71d3-453...,http://www.ville-revin.net/,NaN,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2026-01-23T21:26:01.818Z,NaN,NaN,NaN,[culturalsite],False,False,False,False,False,False,False,False,False,False,False,False,eco,Culture & Musées,False,0.9,"('Culture & Musées', 0.9)",lieu,0.09,lent,0.09,1.062,datatourisme,<NA>,NaN
311729,https://data.datatourisme.fr/42/04338d9e-26e3-...,schema:FoodEstablishment,"Pizzéria ""Annabella""",<NA>,Pizzas à emporter.,Pizza to go.,49.773041,4.722942,France,44,Grand Est,8,Ardennes,8105,Charleville-Mézières,8000,Charleville-Mézières,19 rue du Petit Bois,Spécialités étrangères,Foreign cuisine,None,None,http://cnstlltn.com/1024x768/2dbd53ed-869c-417...,http://cnstlltn.com/1024x768/39ad24af-8bf4-491...,<NA>,NaN,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2026-01-23T21:26:01.818Z,NaN,NaN,NaN,[foodestablishment],True,False,True,False,False,False,False,False,False,False,False,False,eco,Gastronomie & Restauration,False,0.6,"('Gastronomie & Restauration', 0.6)",besoin,0.03,normal,0.06,0.654,datatourisme,<NA>,NaN
311896,https://data.datatourisme.fr/42/13c493a0-096a-...,schema:Accommodation,Le Château de Charbogne,<NA>,Site officiel du château de Charbogne - Meille...,Official website of Château de Charbogne - Bes...,49.502121,4.589075,France,44,Grand Est,8,Ardennes,8103,Charbogne,8130,Charbogne,11 Grande Rue,None,None,None,None,http://cnstlltn.com/1024x768/75bf4358-6e39-4ff...,http://cnstlltn.com/1024x768/8cdfa151-d60b-4f8...,http://www.chateau-charbogne.fr/,NaN,None,None,None,None,None,2020-06-09T00:00:00,2020-07-05T00:00:00,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2026-01-23T21:26:01.818Z,NaN,448.799988,2153.0,[accommodation],False,False,False,False,False,False,False,False,False,False,False,False,eco,Hébergement,False,0.0,"('Hébergement', 0.0)",besoin,0.03,normal,0.06,0.000,datatourisme,<NA>,NaN


<class 'pandas.core.frame.DataFrame'>
Index: 389651 entries, 312421 to 290134
Data columns (total 76 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   source_id                  389651 non-null  string 
 1   poi_types                  389651 non-null  object 
 2   name                       389651 non-null  string 
 3   label_en                   174756 non-null  string 
 4   snippet                    246874 non-null  string 
 5   snippet_en                 248200 non-null  string 
 6   lat                        389651 non-null  float32
 7   lon                        389651 non-null  float32
 8   country                    389651 non-null  string 
 9   region_insee               389651 non-null  Int32  
 10  region                     389651 non-null  string 
 11  dept_insee                 389018 non-null  Int16  
 12  departement                389651 non-null  string 
 13  city_insee                 38

None

In [54]:
display(df_prime_excursion.head(3))
display(df_prime_excursion.info())

,source_id,poi_types,name,label_en,snippet,snippet_en,lat,lon,country,region_insee,region,dept_insee,departement,city_insee,city,postal_code,address_locality,address,theme_fr,theme_en,architectural_style_fr,architectural_style_en,main_media_url,media_resource_url,url,rating,review_value_label_fr,review_value_label_en,sub_category,opens_time,closes_time,hours_valid_from,hours_valid_through,max_people,difficulty_level_fr,locomotion_mode_fr,tour_type_fr,tour_distance_m,duration_min,practice_duration_min,duration_days,practice_duration_days,positive_elevation_gain_m,negative_elevation_loss_m,start_date,end_date,last_update_datatourisme,price,min_price,max_price,type_principal,is_resto_type,is_resto_label,is_resto,is_label_incontournable,is_label_famille,is_label_handicap,is_label_hebergement,is_label_gastronomie,is_label_artisanat,is_label_randonnee,is_label_green,is_etoile,price_level,main_category_compressed,is_prime_plus,main_cat_weight,main_cat_candidates,format_label,format_weight,tempo_label,tempo_weight,score_prime,source,review_count,distance_km
312749,https://data.datatourisme.fr/42/66f5941f-b291-...,olo:OrderedList,Sentier découverte du Vieil-Étang de Bairon,<NA>,"Pédestre - 4,5km - 1h. Faites le tour du Vieil...",Pedestrian - 4.5km - 1h. Take a tour of the Vi...,49.534012,4.769313,France,44,Grand Est,8,Ardennes,8116,Bairon et ses environs,8390,Bairon et ses environs,Digue intermédiaire séparant lac et vieil-étang,Environnement et nature,Nature and environment,None,None,http://cnstlltn.com/1024x768/8e1f1c82-f757-426...,http://cnstlltn.com/master/c2586663-b40c-46f7-...,https://www.cirkwi.com/fr/circuit/242940-varia...,NaN,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,5000.0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2026-01-23T21:26:01.818Z,NaN,NaN,NaN,[orderedlist],False,False,False,False,False,False,False,False,False,False,False,False,eco,Itinéraires & Circuits,True,0.0,"('Itinéraires & Circuits', 0.0)",parcours,0.0,zen,0.0,0.0,datatourisme,<NA>,NaN
313314,https://data.datatourisme.fr/42/9ddfebbd-3e91-...,olo:OrderedList,Captain d'ô douce,<NA>,"Pour égayer vos balades, des prestations telle...","To brighten up your trips, services such as pi...",50.135269,4.823340,France,44,Grand Est,8,Ardennes,8190,Givet,8600,Givet,1 place de la Tour,Bateau électrique,Electric boat,None,None,http://cnstlltn.com/1024x768/c602a48b-2b99-471...,http://cnstlltn.com/1024x768/1af2201b-9adf-44b...,http://www.captaindodouce.fr/,NaN,None,None,None,None,None,2026-04-01T00:00:00,2026-10-31T00:00:00,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2026-01-23T21:26:01.817Z,NaN,50.0,120.0,[orderedlist],False,False,False,False,False,False,False,False,False,False,False,False,eco,Itinéraires & Circuits,True,0.0,"('Itinéraires & Circuits', 0.0)",parcours,0.0,zen,0.0,0.0,datatourisme,<NA>,NaN
313222,https://data.datatourisme.fr/42/94471c27-4ff5-...,olo:OrderedList,À la découverte de Buzancy,<NA>,"Départ du parking de la supérette, rue Dom-Mab...","Departure from the supermarket parking lot, ru...",49.428040,4.955027,France,44,Grand Est,8,Ardennes,8089,Buzancy,8240,Buzancy,<NA>,Histoire,History,None,None,http://cnstlltn.com/1024x768/704eba18-ec22-4ad...,http://cnstlltn.com/1024x768/fddee728-0c25-4bd...,https://www.cirkwi.com/fr/circuit/794509-a-la-...,NaN,None,None,None,None,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,5000.0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,2026-01-23T21:26:01.817Z,NaN,NaN,NaN,[orderedlist],False,False,False,False,False,False,False,False,False,False,False,False,eco,Itinéraires & Circuits,True,0.0,"('Itinéraires & Circuits', 0.0)",parcours,0.0,zen,0.0,0.0,datatourisme,<NA>,NaN


<class 'pandas.core.frame.DataFrame'>
Index: 21983 entries, 312749 to 392139
Data columns (total 76 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   source_id                  21983 non-null  string 
 1   poi_types                  21983 non-null  object 
 2   name                       21983 non-null  string 
 3   label_en                   10043 non-null  string 
 4   snippet                    15693 non-null  string 
 5   snippet_en                 15711 non-null  string 
 6   lat                        21983 non-null  float32
 7   lon                        21983 non-null  float32
 8   country                    21983 non-null  string 
 9   region_insee               21983 non-null  Int32  
 10  region                     21983 non-null  string 
 11  dept_insee                 21960 non-null  Int16  
 12  departement                21983 non-null  string 
 13  city_insee                 21960 non-null  In

None